In [ ]:
!pip install -q opencv-python-headless scikit-learn scipy statsmodels thop


In [ ]:
from __future__ import annotations
import os, sys, gc, ast, csv, io, re, json, math, time, glob, random, warnings, platform, hashlib, traceback
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from typing import Optional
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import sklearn
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "|", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu")


In [ ]:
DATASET = "fia"            # "fia" = Find it again! receipts | "rtm" = RealTextManipulation

# Optional: output folder of the ORIGINAL submitted notebook, attached as a Kaggle input.
# Used only to diagnose the reported attention-localisation AUC (0.325 / 0.577). Leave "" to skip.
OLD_OUT_DIR = {"fia": "/kaggle/input/datasets/smfayaz/docmil-fia/kaggle/working/docmil", 
               "rtm": "/kaggle/input/datasets/smfayaz/docmil-rtm-old/docmil_rtm"}[DATASET]
# Optional: out_dir of the OTHER corpus' run of this notebook (attached as input) for cross-corpus transfer.
OTHER_OUT_DIR = ""


@dataclass
class Config:
    dataset: str = "fia"
    root: str = ""
    out_dir: str = ""
    cache_dir: str = ""            # "" -> out_dir/cache. The cache is deterministic and rebuildable.

    # ---- tiling (annotation-blind) ----
    scales: tuple = (128, 192)
    stride_frac: float = 0.75
    min_overlap: int = 12          # theta_min: forged px^2 inside a tile for a positive tile label
    k_max: int = 24                # K_max: tiles in every evaluation bag
    k_train: int = 48              # blind candidates available to training bags
    k_reserve: int = 8             # extra blind tiles so evidence-ablated bags keep K_max tiles
    k_pos_extra: int = 16          # annotation-positive tiles cached for TRAINING documents only
    bag_size: int = 16             # K_b: tiles per training bag
    cache_seed: int = 1234

    # ---- loss ----
    pos_weight_doc: float = 4.0
    pos_weight_patch: float = 6.0
    focal_gamma: float = 2.0
    aux_patch_weight: float = 0.3

    # ---- optimisation ----
    emb: int = 192                 # embedding width d
    batch_size: int = 8
    epochs: int = 60
    lr_head: float = 1e-3
    lr_backbone: float = 1.5e-4
    weight_decay: float = 1e-4
    freeze_until: int = 2          # MobileNetV3-Small features[0:freeze_until] frozen
    grad_clip: float = 1.0
    amp: bool = True
    num_workers: int = 4

    # ---- protocol ----
    n_folds: int = 5
    split_seed: int = 0            # fold assignment, fixed across seeds
    seeds: tuple = (0, 1, 2)       # tier-1 specs
    seeds_t2: tuple = (0,)
    seeds_ctrl: tuple = (0,)
    seeds_sens: tuple = (0,)
    seeds_lc: tuple = (0,)
    lc_folds: tuple = (0, 1, 2, 3, 4)
    loto_folds: tuple = (0, 1, 2, 3, 4)
    exh_specs: tuple = ("full", "doc_only")
    exh_seeds: tuple = (0, 1, 2)
    sens_base: str = "full"
    lc_fracs: tuple = (0.25, 0.5, 0.75)
    loto_min_forged: int = 10
    tta: bool = True               # 4-view flip TTA at validation-for-threshold and test
    exh_tta: bool = True           # TTA under exhaustive tiling (4x cost); keep equal to tta for a like-for-like comparison
    n_boot: int = 5000
    alpha: float = 0.05

    # ---- whole-image baseline ----
    whole_res: int = 512

    # ---- augmentation ----
    flip_aug: bool = True
    jpeg_aug: bool = True
    jpeg_qf_range: tuple = (70, 100)

    @property
    def k_pool(self):
        return max(self.k_max + self.k_reserve, self.k_train)

    def cache_tag(self):
        return (f"sc{'-'.join(map(str, self.scales))}_st{int(round(100*self.stride_frac))}"
                f"_kp{self.k_pool}_e{self.k_pos_extra}_m{self.min_overlap}_cs{self.cache_seed}")


PRESETS = {
    "fia": dict(dataset="fia",
                root="/kaggle/input/datasets/nikita2998/find-it-again-dataset/findit2",
                out_dir="/kaggle/working/docmil_rev_fia"),
    "rtm": dict(dataset="rtm",
                root="/kaggle/input/datasets/smfayaz/realtextmanip/RealTextManipulation",
                out_dir="/kaggle/working/docmil_rev_rtm",
                k_max=16, k_train=24, k_reserve=8, k_pos_extra=8, bag_size=12,
                pos_weight_doc=1.0, pos_weight_patch=4.0, epochs=15, n_folds=3,
                lc_folds=(0,), loto_folds=(0,), exh_seeds=(0,), whole_res=384,
                # ~12.5 GB tile cache + ~4 GB whole-image cache exceed what fits next to checkpoints in the
                # 20 GB /kaggle/working quota; the cache is deterministic, so rebuild it per session in /tmp
                # (or save it once as a private Kaggle dataset and point cache_dir at the input path).
                cache_dir="/tmp/docmil_rtm_cache"),
}
CFG = Config(**PRESETS[DATASET])
CFG.num_workers = min(CFG.num_workers, os.cpu_count() or 2)

OUT = Path(CFG.out_dir)
CACHE = Path(CFG.cache_dir) if CFG.cache_dir else OUT/"cache"
FIG, TAB, PRED, CKPT, EXH = OUT/"figs", OUT/"tables", OUT/"preds", OUT/"ckpt", OUT/"exh"
for p in (OUT, CACHE, FIG, TAB, PRED, CKPT, EXH, OUT/"logs"):
    p.mkdir(parents=True, exist_ok=True)
LEDGER = OUT/"ledger.jsonl"
AMP_ON = bool(CFG.amp and DEVICE.type == "cuda")
EXH_SHARD = 64
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=1))
print("k_pool =", CFG.k_pool, "| cache tag:", CFG.cache_tag())


In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def load_rgb(p):
    return np.array(Image.open(p).convert("RGB"))


def load_gray(p):
    return np.array(Image.open(p).convert("L"))


def fmt_hms(sec):
    sec = int(max(0, sec)); h, r = divmod(sec, 3600); m, s = divmod(r, 60)
    return f"{h}h{m:02d}m" if h else (f"{m}m{s:02d}s" if m else f"{s}s")


class ETA:
    def __init__(self, total, alpha=.3, skip=1):
        self.total, self.alpha, self.skip = total, alpha, skip
        self.n = 0; self.t0 = self.last = time.time(); self.rate = None

    def step(self, k=1):
        now = time.time(); dt = (now-self.last)/max(k, 1); self.last = now; self.n += k
        if self.n > self.skip:
            self.rate = dt if self.rate is None else (1-self.alpha)*self.rate+self.alpha*dt
        return dt

    @property
    def elapsed(self): return time.time()-self.t0

    @property
    def remaining(self):
        return None if self.rate is None else self.rate*max(0, self.total-self.n)

    def bar(self, w=22):
        f = int(w*min(self.n, self.total)/max(self.total, 1)); r = self.remaining
        return (f"[{'█'*f}{'·'*(w-f)}] {self.n}/{self.total} | {fmt_hms(self.elapsed)}"
                + (f" | ~{fmt_hms(r)} left" if r is not None else ""))


def savefig(fig, name):
    for e in ("pdf", "png"):
        fig.savefig(FIG/f"{name}.{e}", bbox_inches="tight")
    plt.show(); plt.close(fig); print("fig:", name)


def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    return str(o)


def ledger_all():
    out = {}
    if LEDGER.exists():
        for l in LEDGER.read_text().splitlines():
            try: r = json.loads(l)
            except json.JSONDecodeError: continue
            if r.get("status") == "done": out[r["rid"]] = r
    return out


def ledger_get(rid):
    return ledger_all().get(rid)


def ledger_put(rec):
    rec = {**rec, "status": "done", "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(LEDGER, "a") as f:
        f.write(json.dumps(rec, default=_json_default)+"\n")


# ---------------- metrics ----------------
def prf(y, p):
    y = np.asarray(y); p = np.asarray(p)
    tp = int(((p == 1) & (y == 1)).sum()); fp = int(((p == 1) & (y == 0)).sum())
    fn = int(((p == 0) & (y == 1)).sum())
    P = tp/max(tp+fp, 1); R = tp/max(tp+fn, 1)
    return P, R, 2*P*R/max(P+R, 1e-9)


def best_threshold(y, s, w=None):
    """F1-optimal threshold (predict positive if s >= thr). Optional per-sample weights
    (used for prevalence re-weighting). Returns (best F1, threshold)."""
    y = np.asarray(y, float); s = np.asarray(s, float)
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    if len(y) == 0: return 0., .5
    o = np.argsort(-s, kind="mergesort"); ys, ws, ss = y[o], w[o], s[o]
    tp = np.cumsum(ws*ys); fp = np.cumsum(ws*(1-ys)); P = tp[-1]
    if P <= 0: return 0., .5
    last = np.r_[ss[1:] != ss[:-1], True]
    f1 = np.where(last, 2*tp/np.maximum(tp+fp+P, 1e-12), -1.)
    k = int(np.argmax(f1))
    return float(f1[k]), float(ss[k])


def safe_auc(y, s):
    try:
        y = np.asarray(y)
        return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else float("nan")
    except Exception:
        return float("nan")


def safe_ap(y, s):
    try:
        y = np.asarray(y)
        return float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else float("nan")
    except Exception:
        return float("nan")


def auc_rows(S, y):
    """ROC-AUC of every row of S against one label vector (Mann-Whitney, ties averaged)."""
    S = np.atleast_2d(np.asarray(S, float)); y = np.asarray(y).astype(bool)
    n1 = int(y.sum()); n0 = len(y)-n1
    if n1 == 0 or n0 == 0: return np.full(len(S), np.nan)
    r = stats.rankdata(S, axis=1)
    return (r[:, y].sum(1) - n1*(n1+1)/2.)/(n1*n0)


def wilson(k, n, z=1.959964):
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1+z*z/n; c = p+z*z/(2*n); h = z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return (100*(c-h)/d, 100*(c+h)/d)


def holm(p):
    p = np.asarray(p, float); k = len(p); adj = np.full(k, np.nan)
    ok = np.where(np.isfinite(p))[0]
    if not len(ok): return adj
    o = ok[np.argsort(p[ok])]; m = len(o); run = 0.
    for r, i in enumerate(o):
        run = max(run, (m-r)*p[i]); adj[i] = min(run, 1.)
    return adj


def fmt_p(p):
    if p is None or not np.isfinite(p): return "n/a"
    return "<0.001" if p < 1e-3 else f"{p:.3f}"


set_seed(0)
print("utilities ready")


In [ ]:
FORGERY_TYPES = {"CPI", "CPO", "IMI", "CUT", "PIX", "OTHER", "OTHERS"}


def _fia_regions(cell):
    cell = (cell or "").strip()
    if cell in ("0", "", "nan", "None", "0.0"): return []
    try: o = ast.literal_eval(cell)
    except Exception: return []
    return o.get("regions", []) if isinstance(o, dict) else []


def parse_boxes(cell):
    """Genuine forged regions only: drops copy-sources ('Original area' = yes) and
    non-modifications, per the dataset documentation."""
    out = []
    for r in _fia_regions(cell):
        ra = r.get("region_attributes", {})
        if str(ra.get("Original area", "")).strip().lower() == "yes": continue
        mod = ra.get("Modified area", {})
        if not isinstance(mod, dict) or not any(
                str(k).strip().upper() in FORGERY_TYPES and bool(v) for k, v in mod.items()):
            continue
        sa = r.get("shape_attributes", {})
        if sa.get("name") == "rect":
            out.append((int(sa["x"]), int(sa["y"]), int(sa["width"]), int(sa["height"])))
    return out


def parse_types(cell):
    t = []
    for r in _fia_regions(cell):
        ra = r.get("region_attributes", {})
        if str(ra.get("Original area", "")).strip().lower() == "yes": continue
        mod = ra.get("Modified area", {})
        if isinstance(mod, dict):
            t += [str(k).strip().upper() for k, v in mod.items()
                  if str(k).strip().upper() in FORGERY_TYPES and v]
    return sorted(set(t))


def load_fia(root):
    data, listing = {}, []
    for split in ("train", "val", "test"):
        f = Path(root)/f"{split}.txt"
        rd = csv.reader(io.StringIO(f.read_text(encoding="utf-8", errors="replace")))
        h = [x.strip().lower() for x in next(rd)]
        ci, cf = h.index("image"), h.index("forged")
        recs, listed, listed_forged, missing = [], 0, 0, []
        for p in rd:
            if not p or len(p) <= cf: continue
            listed += 1
            name = p[ci].strip(); forged = int(p[cf].strip() == "1"); listed_forged += forged
            fp = Path(root)/split/name
            if not fp.exists(): fp = Path(root)/name
            if not fp.exists():
                missing.append((name, forged)); continue
            ann = p[-1] if len(p) > cf+1 else "0"
            txt = fp.with_suffix(".txt")
            recs.append({"name": name, "rid": Path(name).stem, "path": str(fp), "split": split,
                         "forged": forged, "boxes": parse_boxes(ann) if forged else [],
                         "types": parse_types(ann) if forged else [], "mask_path": None,
                         "prefix": "receipt", "txt": str(txt) if txt.exists() else None,
                         "digital_ann": int(p[1].strip() == "1") if len(p) > 2 else 0,
                         "hand_ann": int(p[2].strip() == "1") if len(p) > 3 else 0})
        data[split] = recs
        listing.append({"split": split, "listed_rows": listed, "listed_forged": listed_forged,
                        "usable": len(recs), "usable_forged": sum(r["forged"] for r in recs),
                        "missing_files": len(missing),
                        "missing_forged": sum(m[1] for m in missing),
                        "missing_names": ";".join(m[0] for m in missing)})
    return data, pd.DataFrame(listing)


def read_stems(p):
    return [s.strip() for s in re.split(r"\s+", Path(p).read_text()) if s.strip()]


def find_file(d, stem, exts=(".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
    for e in exts:
        c = Path(d)/f"{stem}{e}"
        if c.exists(): return str(c)
    return None


def type_of(stem):
    m = re.match(r"^([A-Za-z]+)", stem)
    return m.group(1).lower() if m else "unknown"


def load_rtm(root):
    root = Path(root); img_dir, msk_dir = root/"JPEGImages", root/"SegmentationClass"
    data, listing = {}, []
    for split, fn in (("train", "train.txt"), ("test", "test.txt")):
        stems = read_stems(root/fn); recs, miss = [], 0; eta = ETA(len(stems), skip=0)
        for stem in stems:
            eta.step()
            ip, mp = find_file(img_dir, stem), find_file(msk_dir, stem)
            if ip is None or mp is None: miss += 1; continue
            m = load_gray(mp) > 0; forged = int(m.any()); boxes = []
            if forged:
                _, _, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
                boxes = [(int(x), int(y), int(w), int(h)) for x, y, w, h, a in st[1:] if a >= 4]
            pre = type_of(stem)
            recs.append({"name": Path(ip).name, "rid": stem, "path": ip, "mask_path": mp,
                         "split": split, "forged": forged, "boxes": boxes,
                         "types": [pre] if forged else [], "prefix": pre, "txt": None,
                         "mask_frac": float(m.mean())})
            if eta.n % 2000 == 0: print(f"  {split}: {eta.bar()}")
        data[split] = recs
        listing.append({"split": split, "listed_rows": len(stems), "usable": len(recs),
                        "usable_forged": sum(r["forged"] for r in recs), "missing_files": miss})
    return data, pd.DataFrame(listing)


if CFG.dataset == "fia":
    DATA, LISTING = load_fia(CFG.root)
    POOL_RECS, TEST_RECS = DATA["train"]+DATA["val"], DATA["test"]
else:
    DATA, LISTING = load_rtm(CFG.root)
    POOL_RECS, TEST_RECS = DATA["train"], DATA["test"]

ALL_RECS = POOL_RECS + TEST_RECS
for i, r in enumerate(ALL_RECS): r["di"] = i
POOL_IDX = np.arange(len(POOL_RECS))
TEST_IDX = np.arange(len(POOL_RECS), len(ALL_RECS))
DLABEL = np.array([r["forged"] for r in ALL_RECS], np.float32)
# documents whose tiles may receive auxiliary patch supervision
HAS_ANN = np.array([(r["mask_path"] is not None) if CFG.dataset == "rtm"
                    else (bool(r["forged"]) and len(r["boxes"]) > 0) for r in ALL_RECS], np.float32)

LISTING.to_csv(TAB/"listing.csv", index=False)
rows = []
for s, recs in list(DATA.items()) + [("pool", POOL_RECS), ("test(eval)", TEST_RECS)]:
    f = [r for r in recs if r["forged"]]
    rows.append({"split": s, "docs": len(recs), "authentic": len(recs)-len(f), "forged": len(f),
                 "forged_%": round(100*len(f)/max(len(recs), 1), 1),
                 "forged_with_regions": sum(1 for r in f if r["boxes"]),
                 "forged_without_regions": sum(1 for r in f if not r["boxes"]),
                 "annotated_regions": sum(len(r["boxes"]) for r in f)})
COUNTS = pd.DataFrame(rows); COUNTS.to_csv(TAB/"counts.csv", index=False)
print(LISTING.to_string(index=False)); print(); print(COUNTS.to_string(index=False))

PREV = float(DLABEL[TEST_IDX].mean())
TRIV = {"all_positive_F1": 100*2*PREV/(1+PREV),      # predict forged for every document
        "prevalence_random_F1": 100*PREV,            # random 'forged' with probability = prevalence
        "chance_AUC": 0.5}
print(f"\ntest prevalence {100*PREV:.1f}% | trivial F1: all-positive {TRIV['all_positive_F1']:.1f}, "
      f"prevalence-matched random {TRIV['prevalence_random_F1']:.1f} | chance AUC 0.500")


In [ ]:
_y_pool = DLABEL[POOL_IDX]
FOLDS = [(POOL_IDX[a], POOL_IDX[b]) for a, b in
         StratifiedKFold(CFG.n_folds, shuffle=True, random_state=CFG.split_seed)
         .split(np.zeros(len(POOL_IDX)), _y_pool)]


def _md5(p):
    try: return hashlib.md5(Path(p).read_bytes()).hexdigest()
    except Exception: return None


SPLITS = {"dataset": CFG.dataset, "root": CFG.root, "split_seed": CFG.split_seed,
          "split_files_md5": {fn: _md5(Path(CFG.root)/fn) for fn in
                              (("train.txt", "val.txt", "test.txt") if CFG.dataset == "fia"
                               else ("train.txt", "test.txt"))},
          "pool": [ALL_RECS[d]["rid"] for d in POOL_IDX],
          "test": [ALL_RECS[d]["rid"] for d in TEST_IDX],
          "folds": [{"train": [ALL_RECS[d]["rid"] for d in tr], "val": [ALL_RECS[d]["rid"] for d in va]}
                    for tr, va in FOLDS]}
(OUT/"splits.json").write_text(json.dumps(SPLITS))
for k, (tr, va) in enumerate(FOLDS):
    print(f"fold {k}: train {len(tr)} ({int(DLABEL[tr].sum())} forged) | "
          f"val {len(va)} ({int(DLABEL[va].sum())} forged)")
print("test:", len(TEST_IDX), f"({int(DLABEL[TEST_IDX].sum())} forged)  -> splits.json written")


def permuted_labels(seed):
    """Label-permutation control: document labels shuffled across the whole pool
    (train and validation), test labels untouched."""
    lab = DLABEL.copy(); rng = np.random.default_rng([int(seed), 31337])
    lab[POOL_IDX] = rng.permutation(lab[POOL_IDX]); return lab


In [ ]:
IJG_LUMA = np.array([16, 11, 10, 16, 24, 40, 51, 61, 12, 12, 14, 19, 26, 58, 60, 55, 14, 13, 16, 24,
                     40, 57, 69, 56, 14, 17, 22, 29, 51, 87, 80, 62, 18, 22, 37, 56, 68, 109, 103, 77,
                     24, 35, 55, 64, 81, 104, 113, 92, 49, 64, 78, 87, 103, 121, 120, 101, 72, 92, 95,
                     98, 112, 100, 103, 99], float)


def jpeg_quality(img):
    """IJG quality estimate from the luminance quantisation table (header only)."""
    q = getattr(img, "quantization", None)
    if not q or 0 not in q: return float("nan")
    s = 100.*np.sum(np.array(q[0], float))/IJG_LUMA.sum()
    return float(np.clip((200.-s)/2. if s <= 100 else 5000./s, 1, 100))


def format_audit(recs):
    rows = []
    for r in recs:
        try:
            with Image.open(r["path"]) as im:
                rows.append({"split": r["split"], "forged": r["forged"],
                             "ext": Path(r["path"]).suffix.lower(), "format": im.format,
                             "mode": im.mode, "W": im.size[0], "H": im.size[1],
                             "jpeg_q": jpeg_quality(im) if im.format == "JPEG" else np.nan})
        except Exception as e:
            rows.append({"split": r["split"], "forged": r["forged"],
                         "ext": Path(r["path"]).suffix.lower(), "format": f"ERROR:{type(e).__name__}"})
    return pd.DataFrame(rows)


AUDIT = format_audit(ALL_RECS); AUDIT.to_csv(TAB/"format_audit.csv", index=False)
print("extension x decoded format:\n", pd.crosstab(AUDIT.ext, AUDIT.format), "\n")
print("decoded format x label:\n", pd.crosstab(AUDIT.format, AUDIT.forged), "\n")
print("image size (px):\n", AUDIT.groupby("forged")[["W", "H"]].describe().round(0).T, "\n")
if AUDIT.jpeg_q.notna().any():
    print("estimated JPEG quality by label:\n", AUDIT.groupby("forged").jpeg_q.describe().round(1))
ct = pd.crosstab(AUDIT.format, AUDIT.forged)
if ct.shape[0] > 1 and ct.shape[1] > 1:
    chi = stats.chi2_contingency(ct.values)
    print(f"\nformat vs label chi-square p = {fmt_p(chi[1])}"
          + ("  <-- container format differs by class: a potential shortcut, report it" if chi[1] < .05 else ""))


In [ ]:
def fig_dataset_overview(name=f"fig_{CFG.dataset}_dataset"):
    """Counts come from the same records used for training (R3.5: Fig. 7(c) counts)."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    names = list(DATA.keys())
    fg = [sum(r["forged"] for r in DATA[s]) for s in names]; au = [len(DATA[s])-f for s, f in zip(names, fg)]
    i = np.arange(len(names))
    ax[0].bar(i, au, color="#94a3b8", label="Authentic"); ax[0].bar(i, fg, bottom=au, color="#dc2626", label="Forged")
    for k, (a, f) in enumerate(zip(au, fg)):
        ax[0].text(k, a+f, f"{f}/{a+f}", ha="center", va="bottom", fontsize=7)
    ax[0].set_xticks(i); ax[0].set_xticklabels(names); ax[0].set(ylabel="Documents", title="(a) Class balance")
    ax[0].legend(fontsize=7)
    if CFG.dataset == "rtm":
        v = np.array([100*r["mask_frac"] for r in ALL_RECS if r["forged"]])
        xl = "Tampered pixels (% of document)"
    else:
        v = []
        for r in ALL_RECS:
            if r["forged"] and r["boxes"]:
                try: W, H = Image.open(r["path"]).size
                except Exception: continue
                v.append(100*sum(w*h for (_, _, w, h) in r["boxes"])/(W*H))
        v = np.array(v); xl = "Forged box area (% of document)"
    if v.size:
        ax[1].hist(np.clip(v, 0, np.percentile(v, 99)), bins=40, color="#2563eb")
        ax[1].axvline(np.median(v), color="k", ls="--", lw=1, label=f"median {np.median(v):.3f}%")
        ax[1].set(xlabel=xl, ylabel="Forged documents", title="(b) Forgery extent"); ax[1].legend(fontsize=7)
    tc = Counter(t for r in ALL_RECS if r["forged"] for t in (r["types"] or ["(no region)"]))
    k_, v_ = zip(*sorted(tc.items(), key=lambda z: -z[1]))
    ax[2].bar(k_, v_, color="#0891b2"); ax[2].tick_params(axis="x", rotation=30, labelsize=7)
    for j, c in enumerate(v_): ax[2].text(j, c, str(c), ha="center", va="bottom", fontsize=7)
    ax[2].set(ylabel="Forged documents", title="(c) Technique (documents may have several)")
    savefig(fig, name)


fig_dataset_overview()


In [ ]:
def tile_grid(H, W, ps, stride_frac):
    st = max(8, int(ps*stride_frac))
    ys = list(range(0, max(1, H-ps+1), st)) or [0]
    xs = list(range(0, max(1, W-ps+1), st)) or [0]
    if H > ps and ys[-1] != H-ps: ys.append(H-ps)
    if W > ps and xs[-1] != W-ps: xs.append(W-ps)
    return [(y, x) for y in ys for x in xs]


def ann_mask(rec, H, W):
    """Binary forged-region mask at native resolution, or None. Used ONLY to label tiles."""
    if rec.get("mask_path"):
        m = load_gray(rec["mask_path"])
        if m.shape[:2] != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        return (m > 0).astype(np.uint8)
    if rec["boxes"]:
        m = np.zeros((H, W), np.uint8)
        for (bx, by, bw, bh) in rec["boxes"]:
            m[max(0, by):max(0, by+bh), max(0, bx):max(0, bx+bw)] = 1
        return m
    return None


def _integral(rec, H, W, H2, W2):
    m = ann_mask(rec, H, W)
    if m is None or not m.any(): return None
    mp = np.zeros((H2, W2), np.uint8); mp[:H, :W] = m
    return cv2.integral(mp)


def _ov(ii, y, x, ps):
    if ii is None: return 0
    return int(ii[y+ps, x+ps]-ii[y, x+ps]-ii[y+ps, x]+ii[y, x])


def plan_doc(rec, di, cfg, is_pool):
    try:
        W, H = Image.open(rec["path"]).size
    except Exception:
        return None
    mx = max(cfg.scales); H2, W2 = max(H, mx), max(W, mx)
    # ---- selection: annotation-blind, depends only on (cache_seed, doc index, image size) ----
    rng = np.random.default_rng([cfg.cache_seed, int(di)])
    per = int(math.ceil(cfg.k_pool/len(cfg.scales))); lists = []
    for ps in cfg.scales:
        g = tile_grid(H2, W2, ps, cfg.stride_frac)
        lists.append([(g[i][0], g[i][1], ps) for i in rng.permutation(len(g))[:per]])
    blind = [L[j] for j in range(per) for L in lists if j < len(L)][:cfg.k_pool]
    # ---- labels, computed after selection ----
    ii = _integral(rec, H, W, H2, W2)
    tiles = [(y, x, ps, _ov(ii, y, x, ps), 0, k) for k, (y, x, ps) in enumerate(blind)]
    # ---- annotation-positive extras: training-pool documents only ----
    if is_pool and rec["forged"] and ii is not None and cfg.k_pos_extra > 0:
        seen = set(blind); pos = []
        for ps in cfg.scales:
            for (y, x) in tile_grid(H2, W2, ps, cfg.stride_frac):
                if (y, x, ps) in seen: continue
                a = _ov(ii, y, x, ps)
                if a >= cfg.min_overlap: pos.append((y, x, ps, a))
        if len(pos) > cfg.k_pos_extra:
            r2 = np.random.default_rng([cfg.cache_seed, int(di), 99])
            pos = [pos[i] for i in sorted(r2.choice(len(pos), cfg.k_pos_extra, replace=False))]
        tiles += [(y, x, ps, a, 1, -1) for (y, x, ps, a) in pos]
    return {"H": H, "W": W, "H2": H2, "W2": W2, "tiles": tiles}


def crop_doc(rec, plan, base):
    out = np.full((len(plan["tiles"]), base, base, 3), 255, np.uint8)
    try:
        im = load_rgb(rec["path"])
    except Exception as e:
        print(f"  [warn] unreadable {rec['name']}: {e}"); return out
    if im.shape[:2] != (plan["H"], plan["W"]):
        im = cv2.resize(im, (plan["W"], plan["H"]), interpolation=cv2.INTER_AREA)
    if plan["H2"] > plan["H"] or plan["W2"] > plan["W"]:
        im = cv2.copyMakeBorder(im, 0, plan["H2"]-plan["H"], 0, plan["W2"]-plan["W"],
                                cv2.BORDER_CONSTANT, value=(255, 255, 255))
    for j, (y, x, ps, _, _, _) in enumerate(plan["tiles"]):
        c = im[y:y+ps, x:x+ps]
        out[j] = c if ps == base else cv2.resize(c, (base, base), interpolation=cv2.INTER_AREA)
    return out


def load_cache(d):
    d = Path(d); A = np.load(d/"arrays.npz"); meta = json.loads((d/"meta.json").read_text())
    Z = {k: A[k] for k in A.files}
    Z["patches"] = np.load(d/"patches.npy", mmap_mode="r"); Z["meta"] = meta
    n_docs = len(Z["dlabel"]); doc, src, rank = Z["doc"], Z["src"], Z["rank"]
    bl = np.where(src == 0)[0]; bl = bl[np.lexsort((rank[bl], doc[bl]))]
    ex = np.where(src == 1)[0]; ex = ex[np.argsort(doc[ex], kind="stable")]

    def split(ids):
        cuts = np.searchsorted(doc[ids], np.arange(n_docs+1))
        return [ids[cuts[i]:cuts[i+1]] for i in range(n_docs)]
    Z["blind"], Z["extra"] = split(bl), split(ex)
    return Z


def build_cache(cfg, force=False):
    tag = cfg.cache_tag(); d = CACHE/tag; d.mkdir(parents=True, exist_ok=True)
    meta_p = d/"meta.json"
    if meta_p.exists() and not force and json.loads(meta_p.read_text()).get("complete"):
        print(f"[cache hit] {tag}"); return load_cache(d)
    base = cfg.scales[0]; pool_set = set(POOL_IDX.tolist())
    print(f"[cache] {tag}: planning {len(ALL_RECS)} documents (annotation-blind) ...")
    plans, eta = [], ETA(len(ALL_RECS), skip=0)
    for di, r in enumerate(ALL_RECS):
        plans.append(plan_doc(r, di, cfg, di in pool_set)); eta.step()
        if eta.n % 1000 == 0: print("   ", eta.bar())
    cnt = np.array([0 if p is None else len(p["tiles"]) for p in plans]); off = np.r_[0, np.cumsum(cnt)]
    N = int(off[-1])
    T = np.array([t for p in plans if p for t in p["tiles"]], np.int64).reshape(-1, 6)
    doc = np.repeat(np.arange(len(plans)), cnt).astype(np.int32)
    H2 = np.array([p["H2"] if p else 0 for p in plans]); W2 = np.array([p["W2"] if p else 0 for p in plans])
    y, x, ps = T[:, 0], T[:, 1], T[:, 2]
    xy = np.stack([(x+ps/2)/np.maximum(W2[doc], 1), (y+ps/2)/np.maximum(H2[doc], 1),
                   ps/np.maximum(np.maximum(H2[doc], W2[doc]), 1)], 1).astype(np.float32)
    np.savez(d/"arrays.npz", doc=doc, tyx=T[:, :3].astype(np.int32), ov=T[:, 3].astype(np.int64),
             plabel=(T[:, 3] >= cfg.min_overlap).astype(np.float32), src=T[:, 4].astype(np.int8),
             rank=T[:, 5].astype(np.int32), scale=(ps/base).astype(np.float32), xy=xy,
             dlabel=DLABEL, has_ann=HAS_ANN,
             H=np.array([p["H"] if p else 0 for p in plans]), W=np.array([p["W"] if p else 0 for p in plans]),
             H2=H2, W2=W2)
    pp, prog = d/"patches.npy", d/"progress.json"; done = 0
    if pp.exists() and prog.exists():
        pr = json.loads(prog.read_text()); done = pr.get("done", 0) if pr.get("N") == N else 0
    P = (np.load(pp, mmap_mode="r+") if done else
         np.lib.format.open_memmap(pp, mode="w+", dtype=np.uint8, shape=(max(N, 1), base, base, 3)))
    docs = [di for di in range(len(plans)) if cnt[di] > 0]; CH = 64
    print(f"[cache] cropping {N:,} tiles from {len(docs)} documents (resume at doc {done}) ...")
    eta = ETA(len(docs)-done, skip=0)

    def work(di): return di, crop_doc(ALL_RECS[di], plans[di], base)
    with ThreadPoolExecutor(max(1, CFG.num_workers)) as ex:
        for c0 in range(done, len(docs), CH):
            for di, arr in ex.map(work, docs[c0:c0+CH]):
                P[off[di]:off[di+1]] = arr
            P.flush(); prog.write_text(json.dumps({"N": N, "done": min(c0+CH, len(docs))}))
            eta.step(min(CH, len(docs)-c0))
            if (c0//CH) % 10 == 0: print("   ", eta.bar())
    del P
    meta = {"tag": tag, "complete": True, "N": N, "n_docs": len(plans), "n_pool": int(len(POOL_IDX)),
            "n_test": int(len(TEST_IDX)), "base": base, "dataset": CFG.dataset,
            "cfg": {k: getattr(cfg, k) for k in ("scales", "stride_frac", "min_overlap", "k_max", "k_train",
                                                 "k_reserve", "k_pos_extra", "cache_seed")},
            "k_pool": cfg.k_pool, "blind_tiles": int((T[:, 4] == 0).sum()), "extra_tiles": int((T[:, 4] == 1).sum())}
    meta_p.write_text(json.dumps(meta, indent=1, default=_json_default))
    print(f"[cache] done: {N:,} tiles, {(d/'patches.npy').stat().st_size/1e9:.2f} GB")
    return load_cache(d)


def cache_report(Z, cfg, name="cache_report"):
    """Exact tile counts (R3.1 arithmetic) and how often an annotation-blind evaluation bag
    contains forged evidence at all."""
    rows = []
    for part, idx in (("pool", POOL_IDX), ("test", TEST_IDX)):
        f = [d for d in idx if Z["dlabel"][d] == 1 and Z["has_ann"][d] == 1]
        ev = [Z["blind"][d][:cfg.k_max] for d in f]; trp = [Z["blind"][d][:cfg.k_train] for d in f]
        abl = []
        for d in f:
            b = Z["blind"][d]; abl.append(int((Z["ov"][b] == 0).sum()))
        rows.append({"part": part, "docs": len(idx),
                     "blind_tiles": int(sum(len(Z["blind"][d]) for d in idx)),
                     "extra_positive_tiles": int(sum(len(Z["extra"][d]) for d in idx)),
                     "eval_bag_size_mean": float(np.mean([len(Z["blind"][d][:cfg.k_max]) for d in idx])),
                     "forged_annotated_docs": len(f),
                     "eval_bags_with_forged_tile_%": 100*float(np.mean([(Z["plabel"][h] > 0).any() for h in ev])) if f else np.nan,
                     "eval_bags_with_any_overlap_%": 100*float(np.mean([(Z["ov"][h] > 0).any() for h in ev])) if f else np.nan,
                     "forged_tiles_per_forged_eval_bag": float(np.mean([(Z["plabel"][h] > 0).sum() for h in ev])) if f else np.nan,
                     "train_pools_with_forged_tile_%": 100*float(np.mean([(Z["plabel"][h] > 0).any() for h in trp])) if f else np.nan,
                     "forged_docs_short_after_ablation": int(sum(a < cfg.k_max for a in abl))})
    df = pd.DataFrame(rows); df.to_csv(TAB/f"{name}.csv", index=False)
    print(df.T.to_string(header=False)); return df


_ZCACHE = {}


def get_cache(cfg):
    tag = cfg.cache_tag()
    if tag not in _ZCACHE: _ZCACHE[tag] = build_cache(cfg)
    return _ZCACHE[tag]


Z0 = get_cache(CFG)
CACHE_REPORT = cache_report(Z0, CFG)


In [ ]:
def _jpeg(img01, rng, cfg):
    u = (np.clip(img01, 0, 1)*255).astype(np.uint8)
    q = int(rng.integers(cfg.jpeg_qf_range[0], cfg.jpeg_qf_range[1]+1))
    ok, enc = cv2.imencode(".jpg", cv2.cvtColor(u, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), q])
    if not ok: return img01
    return cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32)/255.


class Bags(Dataset):
    """Document bags built from the annotation-blind candidate list. Uses only arrays in Z."""
    def __init__(self, Z, docs, cfg, train, seed, epoch=0, region=False, evidence="none",
                 labels=None, k_eval=None, bag_size=None):
        self.Z, self.cfg, self.train, self.seed, self.epoch = Z, cfg, train, int(seed), int(epoch)
        self.region, self.evidence = region, evidence
        self.docs = np.asarray(docs, np.int64)
        self.k_eval = k_eval or cfg.k_max; self.bag = bag_size or cfg.bag_size
        self.lab = Z["dlabel"] if labels is None else labels

    def __len__(self): return len(self.docs)

    def pool(self, d, k):
        b = self.Z["blind"][d]; ov = self.Z["ov"]
        if self.evidence == "ablate":                      # drop every tile touching the annotation
            c = b[ov[b] == 0]
            return c[:k] if len(c) else b[:1]
        if self.evidence == "random":                      # drop the same number of random tiles
            head = b[:k]; m = int((ov[head] > 0).sum())
            if m:
                rng = np.random.default_rng([self.cfg.cache_seed, int(d), int(k), 7])
                b = np.delete(b, rng.choice(len(head), m, replace=False))
            return b[:k]
        return b[:k]

    def __getitem__(self, i):
        d = int(self.docs[i]); Z = self.Z
        if self.train:
            rng = np.random.default_rng([self.seed, self.epoch, d])
            base = self.pool(d, self.cfg.k_train)
            if self.region:
                cand = np.concatenate([base, Z["extra"][d]]) if self.evidence == "none" else base
                pl = Z["plabel"][cand]; pos, neg = cand[pl == 1], cand[pl == 0]
                kp = min(len(pos), max(1, self.bag//3)) if len(pos) else 0
                kn = min(self.bag-kp, len(neg)); parts = []
                if kp: parts.append(rng.choice(pos, kp, replace=False))
                if kn: parts.append(rng.choice(neg, kn, replace=False))
                idx = np.concatenate(parts) if parts else base[:1]
            else:
                idx = rng.choice(base, self.bag, replace=False) if len(base) > self.bag else base
        else:
            rng = None; idx = self.pool(d, self.k_eval)
        idx = np.sort(np.asarray(idx, np.int64))
        pats = np.asarray(Z["patches"][idx], dtype=np.float32)/255.
        if self.train:
            for j in range(len(pats)):
                if self.cfg.flip_aug:
                    if rng.random() < .5: pats[j] = pats[j][:, ::-1]
                    if rng.random() < .3: pats[j] = pats[j][::-1]
                if self.cfg.jpeg_aug and rng.random() < .3: pats[j] = _jpeg(pats[j], rng, self.cfg)
        return {"patches": torch.from_numpy(np.ascontiguousarray(pats.transpose(0, 3, 1, 2))),
                "patch_labels": torch.from_numpy(Z["plabel"][idx].astype(np.float32)),
                "scale": torch.from_numpy(Z["scale"][idx].astype(np.float32)),
                "aux_mask": torch.tensor(float(Z["has_ann"][d])),
                "tid": torch.from_numpy(idx),
                "doc_label": torch.tensor(float(self.lab[d])),
                "true_label": torch.tensor(float(Z["dlabel"][d])), "doc_idx": d}


def collate(b):
    return {"patches": torch.cat([x["patches"] for x in b], 0),
            "patch_labels": torch.cat([x["patch_labels"] for x in b], 0),
            "scale": torch.cat([x["scale"] for x in b], 0),
            "aux_mask": torch.cat([x["aux_mask"].repeat(x["patches"].shape[0]) for x in b], 0),
            "tid": torch.cat([x["tid"] for x in b], 0),
            "bag_id": torch.cat([torch.full((x["patches"].shape[0],), i, dtype=torch.long)
                                 for i, x in enumerate(b)]),
            "doc_label": torch.stack([x["doc_label"] for x in b]),
            "true_label": torch.stack([x["true_label"] for x in b]),
            "doc_idx": torch.tensor([x["doc_idx"] for x in b])}


def mil_loader(Z, docs, cfg, train, seed, epoch=0, **kw):
    ds = Bags(Z, docs, cfg, train, seed, epoch, **kw)
    g = torch.Generator(); g.manual_seed(int(seed)*100003+int(epoch))
    return DataLoader(ds, cfg.batch_size if train else 8, shuffle=train, drop_last=train,
                      num_workers=cfg.num_workers, collate_fn=collate, generator=g,
                      pin_memory=DEVICE.type == "cuda", persistent_workers=False)


print("bags ready")


In [ ]:
class PretrainedEncoder(nn.Module):
    """ImageNet MobileNetV3-Small (torchvision IMAGENET1K_V1); features[0:freeze_until] frozen."""
    def __init__(self, freeze_until=2, pretrained=True):
        super().__init__()
        from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
        w = MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        self.f = mobilenet_v3_small(weights=w).features; self.out_ch = 576
        for i, m in enumerate(self.f):
            if i < freeze_until:
                for p in m.parameters(): p.requires_grad = False
        self.register_buffer("mean", torch.tensor([.485, .456, .406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([.229, .224, .225]).view(1, 3, 1, 1))

    def forward(self, x): return self.f((x-self.mean)/self.std)


class AttentionMIL(nn.Module):
    """Gated attention (Ilse et al., 2018): a_k = w^T (tanh(V e_k) ⊙ sigm(U e_k)),
    weights = softmax over the tiles of one bag (Eq. 8). fp32 island."""
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.V = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh())
        self.U = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid())
        self.w = nn.Linear(hidden, 1); self.cls = nn.Linear(dim, 1)

    def forward(self, e, bag, n):
        with torch.autocast(e.device.type, enabled=False):
            e = e.float(); a = self.w(self.V(e)*self.U(e)).squeeze(1)
            out = e.new_zeros(n, e.shape[1]); store = torch.zeros_like(a)
            for b in range(n):
                m = (bag == b)
                if m.sum() == 0: continue
                wt = torch.softmax(a[m], 0); store[m] = wt
                out[b] = (wt.unsqueeze(1)*e[m]).sum(0)
            return self.cls(out).squeeze(1), store


class MaxMIL(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.cls = nn.Linear(dim, 1)

    def forward(self, e, bag, n):
        with torch.autocast(e.device.type, enabled=False):
            e = e.float(); s = self.cls(e).squeeze(1); out = e.new_full((n,), -1e4)
            for b in range(n):
                m = (bag == b)
                if m.sum(): out[b] = s[m].max()
            return out, s


class MeanMIL(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.cls = nn.Linear(dim, 1)

    def forward(self, e, bag, n):
        with torch.autocast(e.device.type, enabled=False):
            e = e.float(); out = e.new_zeros(n, e.shape[1])
            for b in range(n):
                m = (bag == b)
                if m.sum(): out[b] = e[m].mean(0)
            return self.cls(out).squeeze(1), self.cls(e).squeeze(1)


MILS = {"attention": AttentionMIL, "max": MaxMIL, "mean": MeanMIL}


class DocMIL(nn.Module):
    def __init__(self, cfg, mil="attention", pretrained=True):
        super().__init__()
        self.enc = PretrainedEncoder(cfg.freeze_until, pretrained)
        self.proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(self.enc.out_ch, cfg.emb), nn.ReLU(True))
        self.scale_emb = nn.Linear(1, 16)
        self.norm = nn.Sequential(nn.Linear(cfg.emb+16, cfg.emb), nn.ReLU(True), nn.Dropout(.3))
        self.mil = MILS[mil](cfg.emb)
        self.patch_head = nn.Linear(cfg.emb, 1)

    def embed(self, x, scale):
        z = torch.cat([self.proj(self.enc(x)), self.scale_emb(scale.unsqueeze(1).to(x.dtype))], 1)
        return self.norm(z)

    def forward(self, x, bag, n, scale):
        e = self.embed(x, scale); dl, at = self.mil(e, bag, n)
        return dl, self.patch_head(e.float()).squeeze(1), at


class WholeImageCNN(nn.Module):
    """Whole-document baseline: same pretrained trunk, freezing, projection width and
    classifier as DocMIL. The only difference is one downscaled image instead of a bag."""
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.enc = PretrainedEncoder(cfg.freeze_until, pretrained)
        self.proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(self.enc.out_ch, cfg.emb), nn.ReLU(True))
        self.norm = nn.Sequential(nn.Linear(cfg.emb, cfg.emb), nn.ReLU(True), nn.Dropout(.3))
        self.cls = nn.Linear(cfg.emb, 1)

    def forward(self, x): return self.cls(self.norm(self.proj(self.enc(x)))).squeeze(1)


def nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)


def focal(logit, y, pw, gamma, mask=None):
    ce = F.binary_cross_entropy_with_logits(logit, y, reduction="none", pos_weight=pw)
    p = torch.sigmoid(logit); pt = p*y+(1-p)*(1-y); l = (1-pt).pow(gamma)*ce
    if mask is not None:
        l = l*mask; return l.sum()/mask.sum().clamp_min(1.)
    return l.mean()


for k in MILS: print(f"DocMIL[{k:9s}] trainable params: {nparams(DocMIL(CFG, k)):,}")
print(f"WholeImageCNN  trainable params: {nparams(WholeImageCNN(CFG)):,}")
_m = DocMIL(CFG)
with torch.no_grad():
    _o = _m(torch.rand(6, 3, CFG.scales[0], CFG.scales[0]), torch.tensor([0, 0, 0, 1, 1, 1]), 2, torch.ones(6))
print("forward OK:", tuple(_o[0].shape)); del _m, _o; gc.collect()


In [ ]:
@torch.no_grad()
def predict_mil(model, loader, cfg, tta=False):
    model.eval(); views = [(0, 0), (1, 0), (0, 1), (1, 1)] if tta else [(0, 0)]
    amp_on = bool(cfg.amp and DEVICE.type == "cuda"); out = defaultdict(list)
    for b in loader:
        x0 = b["patches"].to(DEVICE, non_blocking=True); bag = b["bag_id"].to(DEVICE)
        sc = b["scale"].to(DEVICE); n = b["doc_label"].shape[0]; acc = 0.
        for vi, (fh, fv) in enumerate(views):
            x = x0.flip(-1) if fh else x0
            if fv: x = x.flip(-2)
            x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(DEVICE.type, enabled=amp_on):
                dl, pl, at = model(x, bag, n, sc)
            acc = acc + torch.sigmoid(dl.float())
            if vi == 0: at0, ph0 = at.float(), torch.sigmoid(pl.float())   # identity view
        bid = b["bag_id"].numpy(); di = b["doc_idx"].numpy()
        out["score"].append((acc/len(views)).cpu().numpy())
        out["ylab"].append(b["doc_label"].numpy()); out["ytrue"].append(b["true_label"].numpy())
        out["doc"].append(di); out["attn"].append(at0.cpu().numpy()); out["phead"].append(ph0.cpu().numpy())
        out["tid"].append(b["tid"].numpy()); out["tdoc"].append(di[bid])
    return {k: np.concatenate(v) for k, v in out.items()}


def mil_loss_fn(sp, cfg):
    aux_w = cfg.aux_patch_weight if sp["aux"] else 0.
    pwd = torch.tensor(cfg.pos_weight_doc, device=DEVICE); pwp = torch.tensor(cfg.pos_weight_patch, device=DEVICE)

    def f(model, b):
        x = b["patches"].to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
        bag = b["bag_id"].to(DEVICE); sc = b["scale"].to(DEVICE); yd = b["doc_label"].to(DEVICE)
        dl, pl, _ = model(x, bag, yd.shape[0], sc)
        loss = focal(dl.float(), yd, pwd, cfg.focal_gamma)
        if aux_w > 0:
            m = b["aux_mask"].to(DEVICE)
            if m.sum() > 0:
                loss = loss + aux_w*focal(pl.float(), b["patch_labels"].to(DEVICE), pwp, cfg.focal_gamma, m)
        return loss
    return f


def train_model(rid, model, make_train_loader, val_loader, loss_fn, predict_fn, cfg):
    """Model selection by validation ROC-AUC (never F1). Resumable per epoch."""
    best_p, last_p = CKPT/f"{rid}.pt", CKPT/f"{rid}.last.pt"
    if best_p.exists():
        try: return torch.load(best_p, map_location="cpu", weights_only=False)
        except Exception: print("  [warn] unreadable checkpoint, retraining")
    model = model.to(DEVICE).to(memory_format=torch.channels_last)
    bb = [p for n_, p in model.named_parameters() if p.requires_grad and n_.startswith("enc.")]
    hd = [p for n_, p in model.named_parameters() if p.requires_grad and not n_.startswith("enc.")]
    opt = torch.optim.AdamW([{"params": bb, "lr": cfg.lr_backbone}, {"params": hd, "lr": cfg.lr_head}],
                            weight_decay=cfg.weight_decay)
    spe = max(1, len(make_train_loader(0)))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, [cfg.lr_backbone, cfg.lr_head],
                                              total_steps=cfg.epochs*spe, pct_start=.2)
    amp_on = bool(cfg.amp and DEVICE.type == "cuda")
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=amp_on)
    start, hist, secs = 0, [], 0.
    best = {"crit": -1., "state": None, "thr": .5, "epoch": -1}
    if last_p.exists():
        try:
            L = torch.load(last_p, map_location="cpu", weights_only=False)
            model.load_state_dict(L["model"]); opt.load_state_dict(L["opt"]); sch.load_state_dict(L["sch"])
            scaler.load_state_dict(L["scaler"])
            start, hist, best, secs = L["epoch"]+1, L["hist"], L["best"], L["secs"]
            print(f"  [resume] {rid} from epoch {start}")
        except Exception as e:
            print(f"  [warn] could not resume ({e}); restarting this run")
    eta = ETA(max(1, cfg.epochs-start)); t0 = time.time()
    for ep in range(start, cfg.epochs):
        model.train(); run = nb = 0
        for b in make_train_loader(ep):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=amp_on):
                loss = loss_fn(model, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update(); sch.step()
            run += float(loss.detach()); nb += 1
        v = predict_fn(model, val_loader, cfg, tta=False)
        auc, ap = safe_auc(v["ylab"], v["score"]), safe_ap(v["ylab"], v["score"])
        f1, thr = best_threshold(v["ylab"], v["score"])
        hist.append({"epoch": ep, "loss": run/max(nb, 1), "val_auc": auc, "val_ap": ap, "val_f1": 100*f1})
        star = ""
        if np.isfinite(auc) and auc > best["crit"]:
            best = {"crit": float(auc), "state": {k: t.detach().cpu().clone() for k, t in model.state_dict().items()},
                    "thr": thr, "epoch": ep}; star = " *"
        dt = eta.step()
        print(f"  ep{ep:02d} {dt:5.1f}s loss={hist[-1]['loss']:.4f} AUC={auc:.3f} AP={ap:.3f} F1={100*f1:.1f}{star}")
        if ep == start or (ep+1) % 5 == 0: print(f"       {eta.bar()}")
        tmp = last_p.with_suffix(".tmp")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sch": sch.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "hist": hist, "best": best,
                    "secs": secs+time.time()-t0}, tmp)
        os.replace(tmp, last_p)
    if best["state"] is None:
        best["state"] = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
    st = {"model": best["state"], "thr": best["thr"], "val_crit": best["crit"], "epoch": best["epoch"],
          "hist": hist, "params": nparams(model), "secs": secs+time.time()-t0}
    torch.save(st, best_p); last_p.unlink(missing_ok=True)
    print(f"  done {fmt_hms(st['secs'])} | best val AUC={best['crit']:.3f} @ epoch {best['epoch']}")
    del opt, sch, scaler; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return st


print("trainer ready")


In [ ]:
def build_whole_cache(cfg, force=False):
    """Aspect-preserving downscale of the whole document into an R x R white canvas."""
    R = cfg.whole_res; d = CACHE/f"whole_r{R}"; d.mkdir(parents=True, exist_ok=True)
    fp, meta_p = d/"images.npy", d/"meta.json"
    if meta_p.exists() and not force and json.loads(meta_p.read_text()).get("complete"):
        print(f"[cache hit] whole_r{R}"); return np.load(fp, mmap_mode="r")
    Wm = np.lib.format.open_memmap(fp, mode="w+", dtype=np.uint8, shape=(len(ALL_RECS), R, R, 3))

    def work(di):
        canvas = np.full((R, R, 3), 255, np.uint8)
        try: im = load_rgb(ALL_RECS[di]["path"])
        except Exception: return di, canvas
        H, W = im.shape[:2]; s = R/max(H, W); h, w = max(1, int(round(H*s))), max(1, int(round(W*s)))
        r_ = cv2.resize(im, (w, h), interpolation=cv2.INTER_AREA)
        y0, x0 = (R-h)//2, (R-w)//2; canvas[y0:y0+h, x0:x0+w] = r_
        return di, canvas
    eta = ETA(len(ALL_RECS), skip=0); CH = 64
    with ThreadPoolExecutor(max(1, CFG.num_workers)) as ex:
        for c0 in range(0, len(ALL_RECS), CH):
            for di, c in ex.map(work, range(c0, min(c0+CH, len(ALL_RECS)))): Wm[di] = c
            eta.step(min(CH, len(ALL_RECS)-c0))
            if (c0//CH) % 10 == 0: print("   ", eta.bar())
    Wm.flush(); del Wm
    meta_p.write_text(json.dumps({"complete": True, "R": R, "n": len(ALL_RECS)}))
    return np.load(fp, mmap_mode="r")


class WholeDS(Dataset):
    def __init__(self, Wc, docs, cfg, train, seed, epoch=0, labels=None):
        self.W, self.cfg, self.train, self.seed, self.epoch = Wc, cfg, train, int(seed), int(epoch)
        self.docs = np.asarray(docs, np.int64); self.lab = DLABEL if labels is None else labels

    def __len__(self): return len(self.docs)

    def __getitem__(self, i):
        d = int(self.docs[i]); im = np.asarray(self.W[d], np.float32)/255.
        if self.train:
            rng = np.random.default_rng([self.seed, self.epoch, d, 5])
            if self.cfg.flip_aug:
                if rng.random() < .5: im = im[:, ::-1]
                if rng.random() < .3: im = im[::-1]
            if self.cfg.jpeg_aug and rng.random() < .3: im = _jpeg(im, rng, self.cfg)
        return {"image": torch.from_numpy(np.ascontiguousarray(im.transpose(2, 0, 1))),
                "doc_label": torch.tensor(float(self.lab[d])),
                "true_label": torch.tensor(float(DLABEL[d])), "doc_idx": d}


def whole_loader(Wc, docs, cfg, train, seed, epoch=0, labels=None):
    g = torch.Generator(); g.manual_seed(int(seed)*100003+int(epoch))
    return DataLoader(WholeDS(Wc, docs, cfg, train, seed, epoch, labels), cfg.batch_size if train else 16,
                      shuffle=train, drop_last=train, num_workers=cfg.num_workers, generator=g,
                      pin_memory=DEVICE.type == "cuda")


def whole_loss_fn(cfg):
    pwd = torch.tensor(cfg.pos_weight_doc, device=DEVICE)

    def f(model, b):
        x = b["image"].to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
        return focal(model(x).float(), b["doc_label"].to(DEVICE), pwd, cfg.focal_gamma)
    return f


@torch.no_grad()
def predict_whole(model, loader, cfg, tta=False):
    model.eval(); views = [(0, 0), (1, 0), (0, 1), (1, 1)] if tta else [(0, 0)]
    amp_on = bool(cfg.amp and DEVICE.type == "cuda"); out = defaultdict(list)
    for b in loader:
        x0 = b["image"].to(DEVICE, non_blocking=True); acc = 0.
        for fh, fv in views:
            x = x0.flip(-1) if fh else x0
            if fv: x = x.flip(-2)
            with torch.autocast(DEVICE.type, enabled=amp_on):
                acc = acc + torch.sigmoid(model(x.contiguous(memory_format=torch.channels_last)).float())
        out["score"].append((acc/len(views)).cpu().numpy()); out["ylab"].append(b["doc_label"].numpy())
        out["ytrue"].append(b["true_label"].numpy()); out["doc"].append(b["doc_idx"].numpy())
    return {k: np.concatenate(v) for k, v in out.items()}


_WCACHE = {}


def get_whole(cfg):
    if cfg.whole_res not in _WCACHE: _WCACHE[cfg.whole_res] = build_whole_cache(cfg)
    return _WCACHE[cfg.whole_res]


print("whole-image baseline ready")


In [ ]:
BASE_SPEC = dict(kind="mil", mil="attention", pretrained=True, region=True, aux=True, evidence="none",
                 permute=False, train_frac=1.0, holdout_type=None, cfg_over={})
SPECS = {
    # ---- supervision ablation (R2): 2 x 2 over {region-aware sampling} x {auxiliary instance loss}
    "full":            {},                                    # region sampling + aux loss (submitted model)
    "sampling_only":   dict(aux=False),
    "aux_only":        dict(region=False),
    "doc_only":        dict(region=False, aux=False),         # strict document labels only
    # ---- aggregation (R2.4, R3.10)
    "full_mean":       dict(mil="mean"),
    "full_max":        dict(mil="max"),
    "doc_only_mean":   dict(region=False, aux=False, mil="mean"),
    "doc_only_max":    dict(region=False, aux=False, mil="max"),
    # ---- transfer
    "no_pretrain":     dict(pretrained=False),
    # ---- baseline (R2.3, R3.7)
    "whole_image":     dict(kind="whole", region=False, aux=False),
    # ---- controls (R3.2), all on the strict document-only configuration
    "doc_only_evid":   dict(region=False, aux=False, evidence="ablate"),
    "doc_only_randrm": dict(region=False, aux=False, evidence="random"),
    "doc_only_perm":   dict(region=False, aux=False, permute=True),
    "full_perm":       dict(permute=True),
}
TIER1 = ["full", "doc_only", "sampling_only", "aux_only", "full_mean", "whole_image"]
TIER2 = ["full_max", "doc_only_mean", "doc_only_max", "no_pretrain"]
CONTROLS = ["doc_only_evid", "doc_only_randrm", "doc_only_perm", "full_perm"]


def get_spec(name):
    if name in SPECS:
        s = {**BASE_SPEC, **SPECS[name]}; s["cfg_over"] = dict(s["cfg_over"]); s["name"] = name; return s
    if "@" in name:
        base, mod = name.split("@", 1); s = get_spec(base); s["name"] = name
        if mod.startswith("frac"): s["train_frac"] = float(mod[4:])
        elif mod.startswith("loto_"): s["holdout_type"] = mod[5:]
        else: raise KeyError(name)
        return s
    if name.startswith("sens_"):
        _, key, val = name.split("_", 2); s = get_spec(CFG.sens_base); s["name"] = name
        if key == "scales": s["cfg_over"]["scales"] = tuple(int(v) for v in val.split("-"))
        elif key == "stride": s["cfg_over"]["stride_frac"] = int(val)/100
        elif key == "bag": s["cfg_over"]["bag_size"] = int(val)
        else: raise KeyError(name)
        return s
    raise KeyError(name)


def sens_specs():
    out = ["sens_scales_"+"-".join(map(str, sc)) for sc in [(128,), (192,), (128, 192, 256)] if sc != CFG.scales]
    out += [f"sens_stride_{int(100*st)}" for st in (0.5, 1.0) if st != CFG.stride_frac]
    out += [f"sens_bag_{b}" for b in sorted({max(4, CFG.bag_size//2), min(CFG.k_max, CFG.bag_size+CFG.bag_size//2)})
            if b != CFG.bag_size]
    return out


def lc_specs():
    return [f"{CFG.sens_base}@frac{f}" for f in CFG.lc_fracs]


def loto_types():
    cp = Counter(t for d in POOL_IDX if DLABEL[d] == 1 for t in set(ALL_RECS[d]["types"]))
    ct = Counter(t for d in TEST_IDX if DLABEL[d] == 1 for t in set(ALL_RECS[d]["types"]))
    return [t for t, n in sorted(cp.items()) if n >= CFG.loto_min_forged and ct.get(t, 0) >= 5]


def loto_specs():
    return [f"{CFG.sens_base}@loto_{t}" for t in loto_types()]


def filter_docs(docs, sp, seed, is_train):
    docs = np.asarray(docs, np.int64); t = sp.get("holdout_type")
    if t:   # held-out technique removed from training AND validation (no selection leakage)
        docs = np.array([d for d in docs if not (DLABEL[d] == 1 and t in ALL_RECS[d]["types"])], np.int64)
    if is_train and sp["train_frac"] < 1:
        rng = np.random.default_rng([int(seed), 4242, int(1000*sp["train_frac"])]); keep = []
        for c in (0, 1):
            ids = docs[DLABEL[docs] == c]
            if len(ids): keep.append(rng.choice(ids, max(1, int(round(sp["train_frac"]*len(ids)))), replace=False))
        docs = np.sort(np.concatenate(keep))
    return docs


def test_docs_for(sp):
    t = sp.get("holdout_type")
    if not t: return TEST_IDX
    return np.array([d for d in TEST_IDX if DLABEL[d] == 0 or t in ALL_RECS[d]["types"]], np.int64)


def run_one(name, seed, fold):
    rid = f"{name}__s{seed}__f{fold}"
    done = ledger_get(rid)
    if done: return done
    sp = get_spec(name); cfg = replace(CFG, **sp["cfg_over"]) if sp["cfg_over"] else CFG
    tr, va = FOLDS[fold]
    tr, va = filter_docs(tr, sp, seed, True), filter_docs(va, sp, seed, False)
    te = test_docs_for(sp); labels = permuted_labels(seed) if sp["permute"] else None
    print(f"\n--- {rid} | train={len(tr)} ({int(DLABEL[tr].sum())} forged) val={len(va)} test={len(te)} ---")
    set_seed(seed)
    if sp["kind"] == "mil":
        Z = get_cache(cfg)
        mk = lambda ep: mil_loader(Z, tr, cfg, True, seed, ep, region=sp["region"],
                                   evidence=sp["evidence"], labels=labels)
        vl = mil_loader(Z, va, cfg, False, seed, 0, evidence=sp["evidence"], labels=labels)
        st = train_model(rid, DocMIL(cfg, sp["mil"], sp["pretrained"]), mk, vl, mil_loss_fn(sp, cfg), predict_mil, cfg)
        model = DocMIL(cfg, sp["mil"], sp["pretrained"]).to(DEVICE)
        model.load_state_dict(st["model"]); model = model.to(memory_format=torch.channels_last)
        v = predict_mil(model, vl, cfg, tta=cfg.tta)
        t = predict_mil(model, mil_loader(Z, te, cfg, False, seed, 0, evidence=sp["evidence"]), cfg, tta=cfg.tta)
    else:
        Wc = get_whole(cfg)
        mk = lambda ep: whole_loader(Wc, tr, cfg, True, seed, ep, labels)
        vl = whole_loader(Wc, va, cfg, False, seed, 0, labels)
        st = train_model(rid, WholeImageCNN(cfg, sp["pretrained"]), mk, vl, whole_loss_fn(cfg), predict_whole, cfg)
        model = WholeImageCNN(cfg, sp["pretrained"]).to(DEVICE)
        model.load_state_dict(st["model"]); model = model.to(memory_format=torch.channels_last)
        v = predict_whole(model, vl, cfg, tta=cfg.tta)
        t = predict_whole(model, whole_loader(Wc, te, cfg, False, seed, 0), cfg, tta=cfg.tta)
    _, thr = best_threshold(v["ylab"], v["score"])          # validation threshold, same TTA protocol as test
    y, s = t["ytrue"], t["score"]; P, R, F1 = prf(y, (s >= thr).astype(int))
    rec = {"rid": rid, "name": name, "seed": int(seed), "fold": int(fold), "auc": safe_auc(y, s),
           "ap": safe_ap(y, s), "P": 100*P, "R": 100*R, "F1": 100*F1, "thr": float(thr),
           "val_auc": st["val_crit"], "best_epoch": st["epoch"], "params": st["params"],
           "secs": st["secs"], "n_train": len(tr), "n_test": len(y), "n_test_pos": int(y.sum())}
    np.savez_compressed(PRED/f"{rid}.npz", thr=thr, **{"t_"+k: a for k, a in t.items()},
                        **{"v_"+k: a for k, a in v.items()})
    ledger_put(rec)
    print(f"  TEST AUC={rec['auc']:.3f} AP={rec['ap']:.3f} F1={rec['F1']:.1f} (P={rec['P']:.1f} R={rec['R']:.1f}) thr={thr:.3f}")
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return rec


def run_spec(name, seeds=None, folds=None):
    seeds = CFG.seeds if seeds is None else seeds
    folds = range(CFG.n_folds) if folds is None else folds
    jobs = [(s, k) for s in seeds for k in folds]; L = ledger_all()
    todo = [j for j in jobs if f"{name}__s{j[0]}__f{j[1]}" not in L]
    print(f"\n{'='*70}\n{name}: {len(todo)}/{len(jobs)} runs to do\n{'='*70}")
    out = []
    for s, k in jobs:
        try:
            out.append(run_one(name, s, k))
        except Exception as e:
            print(f"[FAIL] {name} s{s} f{k}: {type(e).__name__}: {e}"); traceback.print_exc()
    df = pd.DataFrame(out)
    if len(df):
        print(f"=== {name}: {len(df)} runs | AUC {df.auc.mean():.3f} ± {df.auc.std(ddof=1) if len(df) > 1 else 0:.3f} "
              f"| F1 {df.F1.mean():.1f} ± {df.F1.std(ddof=1) if len(df) > 1 else 0:.1f}")
    return df


def planned_jobs():
    q = [(n, CFG.seeds, tuple(range(CFG.n_folds))) for n in TIER1]
    q += [(n, CFG.seeds_t2, tuple(range(CFG.n_folds))) for n in TIER2]
    q += [(n, CFG.seeds_ctrl, tuple(range(CFG.n_folds))) for n in CONTROLS]
    q += [(n, CFG.seeds_sens, tuple(range(CFG.n_folds))) for n in sens_specs()]
    q += [(n, CFG.seeds_lc, CFG.lc_folds) for n in lc_specs()]
    q += [(n, CFG.seeds_lc, CFG.loto_folds) for n in loto_specs()]
    return q


def show_plan():
    L = ledger_all()
    secs = [r["secs"] for r in L.values() if r["name"] in ("full", "doc_only", "sampling_only", "aux_only")]
    per = float(np.median(secs)) if secs else None; rows = []
    for n, ss, ff in planned_jobs():
        tot = len(ss)*len(ff); done = sum(1 for s in ss for k in ff if f"{n}__s{s}__f{k}" in L)
        rows.append({"spec": n, "seeds": str(tuple(ss)), "folds": len(ff), "runs": tot, "done": done, "todo": tot-done})
    df = pd.DataFrame(rows); print(df.to_string(index=False))
    todo = int(df.todo.sum())
    print(f"\n{todo} MIL/whole runs left" + (f" ≈ {todo*per/3600:.1f} GPU-h at the measured {fmt_hms(per)}/run" if per
          else " (time one 'full' run, then re-run this cell for an estimate)"))
    print("Priority: Tier 1 -> controls -> exhaustive -> Tier 2 -> sensitivity -> learning curve / LOTO.")
    return df


PLAN = show_plan()


In [ ]:
# Tier 1: supervision ablation (R2), attention vs mean (R2.4/R3.10), whole-image baseline (R2.3/R3.7)
for n in TIER1:
    run_spec(n, CFG.seeds)


In [ ]:
# Controls (R3.2): evidence ablation, matched random removal, label permutation
for n in CONTROLS:
    run_spec(n, CFG.seeds_ctrl)


@torch.no_grad()
def eval_only(base, mode, seeds=None, folds=None):
    """Test-time-only evidence removal on models trained normally (no retraining)."""
    name = f"{base}+ev{mode}"; sp = get_spec(base); Z = get_cache(CFG)
    seeds = CFG.seeds if seeds is None else seeds; folds = range(CFG.n_folds) if folds is None else folds
    L = ledger_all()
    for s in seeds:
        for k in folds:
            rid, brid = f"{name}__s{s}__f{k}", f"{base}__s{s}__f{k}"
            if rid in L or brid not in L or not (CKPT/f"{brid}.pt").exists(): continue
            st = torch.load(CKPT/f"{brid}.pt", map_location="cpu", weights_only=False)
            model = DocMIL(CFG, sp["mil"], sp["pretrained"]).to(DEVICE); model.load_state_dict(st["model"])
            model = model.to(memory_format=torch.channels_last)
            t = predict_mil(model, mil_loader(Z, TEST_IDX, CFG, False, s, 0, evidence=mode), CFG, tta=CFG.tta)
            thr = L[brid]["thr"]; y, sc = t["ytrue"], t["score"]; P, R, F1 = prf(y, (sc >= thr).astype(int))
            rec = {"rid": rid, "name": name, "seed": s, "fold": k, "auc": safe_auc(y, sc), "ap": safe_ap(y, sc),
                   "P": 100*P, "R": 100*R, "F1": 100*F1, "thr": thr, "params": L[brid]["params"],
                   "secs": 0., "n_test": len(y), "n_test_pos": int(y.sum())}
            np.savez_compressed(PRED/f"{rid}.npz", thr=thr, **{"t_"+kk: a for kk, a in t.items()})
            ledger_put(rec); print(f"{rid}: AUC={rec['auc']:.3f} (base {L[brid]['auc']:.3f})")
            del model; gc.collect()


for base in ("full", "doc_only"):
    for mode in ("ablate", "random"):
        eval_only(base, mode)


In [ ]:
def exh_shards():
    return [TEST_IDX[i:i+EXH_SHARD] for i in range(0, len(TEST_IDX), EXH_SHARD)]


def full_tiles(rec, cfg):
    im = load_rgb(rec["path"]); H, W = im.shape[:2]; mx = max(cfg.scales); H2, W2 = max(H, mx), max(W, mx)
    if (H2, W2) != (H, W):
        im = cv2.copyMakeBorder(im, 0, H2-H, 0, W2-W, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    meta = [(y, x, ps) for ps in cfg.scales for (y, x) in tile_grid(H2, W2, ps, cfg.stride_frac)]
    base = cfg.scales[0]; crops = np.empty((len(meta), base, base, 3), np.uint8)
    for i, (y, x, ps) in enumerate(meta):
        c = im[y:y+ps, x:x+ps]
        crops[i] = c if ps == base else cv2.resize(c, (base, base), interpolation=cv2.INTER_AREA)
    ii = _integral(rec, H, W, H2, W2)
    ov = np.array([_ov(ii, y, x, ps) for (y, x, ps) in meta], np.int64)
    return im, meta, crops, ov


@torch.no_grad()
def doc_forward_full(mdl, crops, meta, cfg, chunk=256):
    base = cfg.scales[0]; views = [(0, 0), (1, 0), (0, 1), (1, 1)] if cfg.exh_tta else [(0, 0)]
    sc_all = torch.tensor([ps/base for _, _, ps in meta], dtype=torch.float32, device=DEVICE)
    zero = None; prob = 0.
    for vi, (fh, fv) in enumerate(views):
        E = []
        for c0 in range(0, len(meta), chunk):
            x = torch.from_numpy(crops[c0:c0+chunk]).to(DEVICE).permute(0, 3, 1, 2).float().div_(255.)
            if fh: x = x.flip(-1)
            if fv: x = x.flip(-2)
            with torch.autocast(DEVICE.type, enabled=AMP_ON):
                E.append(mdl.embed(x.contiguous(memory_format=torch.channels_last), sc_all[c0:c0+chunk]).float())
        E = torch.cat(E, 0)
        if zero is None: zero = torch.zeros(len(E), dtype=torch.long, device=DEVICE)
        dl, at = mdl.mil(E, zero, 1); prob += float(torch.sigmoid(dl.float())[0])
        if vi == 0:
            a0 = at.float().cpu().numpy(); p0 = torch.sigmoid(mdl.patch_head(E).float()).squeeze(1).cpu().numpy()
    return prob/len(views), a0, p0


@torch.no_grad()
def exhaustive_eval(names=None, seeds=None):
    names = CFG.exh_specs if names is None else names; seeds = CFG.exh_seeds if seeds is None else seeds
    rids = [f"{n}__s{s}__f{k}" for n in names for s in seeds for k in range(CFG.n_folds)
            if (CKPT/f"{n}__s{s}__f{k}.pt").exists()]
    shards = exh_shards()
    todo = {rid: [j for j in range(len(shards)) if not (EXH/rid/f"shard{j:04d}.npz").exists()] for rid in rids}
    need = sorted(set(j for v in todo.values() for j in v))
    print(f"[exhaustive] {len(rids)} models, {len(need)}/{len(shards)} shards to process")
    if not need: return
    models = {}
    for rid in rids:
        if not todo[rid]: continue
        sp = get_spec(rid.split("__")[0])
        st = torch.load(CKPT/f"{rid}.pt", map_location="cpu", weights_only=False)
        m = DocMIL(CFG, sp["mil"], sp["pretrained"]).to(DEVICE).eval(); m.load_state_dict(st["model"])
        models[rid] = m.to(memory_format=torch.channels_last); (EXH/rid).mkdir(parents=True, exist_ok=True)
    eta = ETA(len(need), skip=0)
    for j in need:
        buf = {rid: defaultdict(list) for rid in models if j in todo[rid]}
        for d in shards[j]:
            _, meta, crops, ov = full_tiles(ALL_RECS[d], CFG)
            for rid, B in buf.items():
                p, a, ph = doc_forward_full(models[rid], crops, meta, CFG)
                B["doc"].append(d); B["score"].append(p); B["n"].append(len(meta)); B["attn"].append(a)
                B["phead"].append(ph); B["ov"].append(ov); B["tyx"].append(np.array(meta, np.int32))
        for rid, B in buf.items():
            np.savez_compressed(EXH/rid/f"shard{j:04d}.npz", doc=np.array(B["doc"]),
                                score=np.array(B["score"], np.float32), n=np.array(B["n"]),
                                attn=np.concatenate(B["attn"]).astype(np.float32),
                                phead=np.concatenate(B["phead"]).astype(np.float32),
                                ov=np.concatenate(B["ov"]).astype(np.int32), tyx=np.concatenate(B["tyx"]))
        eta.step(); print(f"  shard {j}: {eta.bar()}")
    del models; gc.collect()


def load_exh(rid):
    shards = exh_shards(); fs = [EXH/rid/f"shard{j:04d}.npz" for j in range(len(shards))]
    if not all(f.exists() for f in fs): return None
    parts = [np.load(f) for f in fs]
    out = {k: np.concatenate([p[k] for p in parts]) for k in ("doc", "score", "n", "attn", "phead", "ov", "tyx")}
    out["tdoc"] = np.repeat(out["doc"], out["n"]); out["off"] = np.r_[0, np.cumsum(out["n"])]
    return out


exhaustive_eval()


In [ ]:
# Tier 2 (aggregation and transfer ablations)
for n in TIER2:
    run_spec(n, CFG.seeds_t2)
# Sensitivity: scale set, stride, bag size (R4.3). One factor at a time around CFG.
for n in sens_specs():
    run_spec(n, CFG.seeds_sens)
# Data scaling (R1.4) and leave-one-technique-out generalisation to unseen forgery types (R1.2)
for n in lc_specs():
    run_spec(n, CFG.seeds_lc, CFG.lc_folds)
print("LOTO techniques:", loto_types())
for n in loto_specs():
    run_spec(n, CFG.seeds_lc, CFG.loto_folds)


In [ ]:
PAPER_BASELINES = [   # Martínez Tornés et al. (ICDAR 2023), official split, Forged class; no ROC-AUC reported
    {"model": "Text (BoW+LR)†", "P": 40.0, "R": 11.4, "F1": 17.8},
    {"model": "Image (SVM)†", "P": 30.0, "R": 8.6, "F1": 13.3},
    {"model": "ChatGPT strict†", "P": 14.7, "R": 88.6, "F1": 25.2},
    {"model": "ChatGPT relaxed†", "P": 18.3, "R": 62.9, "F1": 28.4},
] if CFG.dataset == "fia" else []


def baseline_pixel_svm(size=250):
    """Reproduction of the published pixel-SVM baseline on the official train/test split."""
    from sklearn.svm import LinearSVC

    def feats(rows):
        X, y = [], []
        for r in rows:
            try: g = cv2.cvtColor(load_rgb(r["path"]), cv2.COLOR_RGB2GRAY)
            except Exception: continue
            X.append(cv2.resize(g, (size, size)).ravel().astype(np.float32)/255.); y.append(r["forged"])
        return np.array(X), np.array(y)
    Xtr, ytr = feats(DATA["train"]); Xte, yte = feats(DATA["test"])
    sc = StandardScaler().fit(Xtr)
    clf = LinearSVC(class_weight="balanced", max_iter=5000).fit(sc.transform(Xtr), ytr)
    return yte, clf.predict(sc.transform(Xte)), clf.decision_function(sc.transform(Xte))


def baseline_text_lr():
    from sklearn.feature_extraction.text import CountVectorizer

    def load(rows):
        T, y = [], []
        for r in rows:
            if not r.get("txt"): continue
            T.append(Path(r["txt"]).read_text(encoding="utf-8", errors="replace")); y.append(r["forged"])
        return T, np.array(y)
    Ttr, ytr = load(DATA["train"]); Tte, yte = load(DATA["test"])
    if not Ttr or not Tte: return None
    v = CountVectorizer(max_features=20000, ngram_range=(1, 2)).fit(Ttr)
    c = LogisticRegression(max_iter=2000, class_weight="balanced").fit(v.transform(Ttr), ytr)
    return yte, c.predict(v.transform(Tte)), c.predict_proba(v.transform(Tte))[:, 1]


CLASSICAL = []
if CFG.dataset == "fia":
    for nm, fn in [("Pixel SVM (reproduced)", baseline_pixel_svm), ("Text BoW+LR (reproduced)", baseline_text_lr)]:
        try:
            o = fn()
            if o is None: print(nm, "skipped"); continue
            y, p, s = o; P, R, F1 = prf(y, p)
            CLASSICAL.append({"model": nm, "P": 100*P, "R": 100*R, "F1": 100*F1, "ROC-AUC": safe_auc(y, s),
                              "PR-AUC": safe_ap(y, s)})
        except Exception as e:
            print(nm, "failed:", e)
    pd.DataFrame(CLASSICAL).to_csv(TAB/"classical.csv", index=False)
    print(pd.DataFrame(CLASSICAL).round(3).to_string(index=False))


def cv_sklearn(X, make_clf, name):
    """Same folds as DocMIL; threshold from out-of-fold pool predictions; test = mean of fold models."""
    oof = np.full(len(ALL_RECS), np.nan); te = []
    for tr, va in FOLDS:
        clf = make_clf().fit(X[tr], DLABEL[tr])
        oof[va] = clf.predict_proba(X[va])[:, 1]; te.append(clf.predict_proba(X[TEST_IDX])[:, 1])
    s = np.mean(te, 0); y = DLABEL[TEST_IDX].astype(int)
    thr = best_threshold(DLABEL[POOL_IDX], oof[POOL_IDX])[1]; P, R, F1 = prf(y, (s >= thr).astype(int))
    np.savez(PRED/f"sk_{name}.npz", oof=oof, test=s, test_folds=np.array(te), thr=thr)
    fold_auc = auc_rows(np.array(te), y)
    return {"model": name, "P": 100*P, "R": 100*R, "F1": 100*F1, "ROC-AUC": safe_auc(y, s),
            "PR-AUC": safe_ap(y, s), "fold_mean_AUC": float(np.mean(fold_auc)),
            "fold_sd_AUC": float(np.std(fold_auc, ddof=1)) if len(te) > 1 else np.nan}


def _dct_mat():
    C = np.zeros((8, 8))
    for k in range(8):
        a = math.sqrt(1/8) if k == 0 else math.sqrt(2/8)
        for n in range(8): C[k, n] = a*math.cos(math.pi*(2*n+1)*k/16)
    return C.astype(np.float32)


DCT8 = _dct_mat()
ZIGZAG = [i*8+j for i, j in sorted(((i, j) for i in range(8) for j in range(8)),
                                   key=lambda p: (p[0]+p[1], p[0] if (p[0]+p[1]) % 2 else p[1]))]


def dct_fd_features(path, n_modes=20):
    with Image.open(path) as im:
        Y = np.asarray(im.convert("RGB").convert("YCbCr"))[..., 0].astype(np.float32)-128.
    H8, W8 = Y.shape[0]//8*8, Y.shape[1]//8*8
    B = Y[:H8, :W8].reshape(H8//8, 8, W8//8, 8).transpose(0, 2, 1, 3).reshape(-1, 8, 8)
    D = (DCT8 @ B @ DCT8.T).reshape(-1, 64)[:, ZIGZAG[1:n_modes+1]]
    f = []
    for m in range(n_modes):
        v = np.abs(np.round(D[:, m])); v = v[v >= 1]
        if not len(v): f += [0.]*9; continue
        fd = np.clip((v/10**np.floor(np.log10(v))).astype(int), 1, 9)
        h = np.bincount(fd, minlength=10)[1:].astype(float); f += list(h/h.sum())
    return np.array(f, np.float32)


fp_dct = OUT/"dct_fd_features.npy"
if fp_dct.exists():
    XDCT = np.load(fp_dct)
else:
    XDCT = np.zeros((len(ALL_RECS), 180), np.float32); eta = ETA(len(ALL_RECS), skip=0)

    def _w(di):
        try: return di, dct_fd_features(ALL_RECS[di]["path"])
        except Exception: return di, np.zeros(180, np.float32)
    with ThreadPoolExecutor(max(1, CFG.num_workers)) as ex:
        for c0 in range(0, len(ALL_RECS), 64):
            for di, f in ex.map(_w, range(c0, min(c0+64, len(ALL_RECS)))): XDCT[di] = f
            eta.step(min(64, len(ALL_RECS)-c0))
            if (c0//64) % 20 == 0: print("  DCT features:", eta.bar())
    np.save(fp_dct, XDCT)
DCT_ROW = cv_sklearn(XDCT, lambda: make_pipeline(StandardScaler(), LogisticRegression(
    C=0.1, max_iter=3000, class_weight="balanced")), "dct_first_digit_lr")
pd.DataFrame([DCT_ROW]).to_csv(TAB/"dct_baseline.csv", index=False)
print(pd.DataFrame([DCT_ROW]).round(3).to_string(index=False))
if "format" in AUDIT:
    jpg = (AUDIT.format == "JPEG").values[TEST_IDX]
    if 0 < jpg.sum() < len(jpg):
        s = np.load(PRED/"sk_dct_first_digit_lr.npz")["test"]; y = DLABEL[TEST_IDX]
        print(f"DCT baseline AUC on JPEG-stored test docs only: {safe_auc(y[jpg], s[jpg]):.3f} (n={int(jpg.sum())})")


In [ ]:
def load_runs(name, source="bag", seeds=None):
    L = ledger_all()
    recs = sorted([r for r in L.values() if r["name"] == name and (seeds is None or r["seed"] in seeds)],
                  key=lambda r: (r["seed"], r["fold"]))
    S, keep, D = [], [], None
    for r in recs:
        if source == "bag":
            f = PRED/f"{r['rid']}.npz"
            if not f.exists(): continue
            z = np.load(f); s, dd = z["t_score"], z["t_doc"]
        else:
            e = load_exh(r["rid"])
            if e is None: continue
            s, dd = e["score"], e["doc"]
        if D is None: D = dd
        elif not np.array_equal(D, dd): print(f"[warn] {r['rid']}: test order differs, skipped"); continue
        S.append(s); keep.append(r)
    if not keep: return None
    return keep, np.stack(S), DLABEL[D].astype(int), D


def _sd(v):
    v = np.asarray(v, float); return float(np.std(v, ddof=1)) if len(v) > 1 else float("nan")


def summarize(name, source="bag", seeds=None, docs=None):
    Lr = load_runs(name, source, seeds)
    if Lr is None: return None
    recs, S, y, D = Lr
    if docs is not None:
        m = np.isin(D, docs); S, y, D = S[:, m], y[m], D[m]
    aucs = auc_rows(S, y); rows = []
    for r, s, a in zip(recs, S, aucs):
        P, R, F1 = prf(y, (s >= r["thr"]).astype(int))
        rows.append({"seed": r["seed"], "fold": r["fold"], "auc": a, "ap": safe_ap(y, s),
                     "F1": 100*F1, "P": 100*P, "R": 100*R, "thr": r["thr"]})
    df = pd.DataFrame(rows)
    row = {"spec": name, "source": source, "n_models": len(df), "n_seeds": df.seed.nunique(),
           "n_test": len(y), "n_test_forged": int(y.sum()),
           "auc": df.auc.mean(), "auc_sd": _sd(df.auc), "ap": df.ap.mean(), "ap_sd": _sd(df.ap),
           "F1": df.F1.mean(), "F1_sd": _sd(df.F1), "P": df.P.mean(), "R": df.R.mean(),
           "params": recs[0].get("params")}
    E = []
    for _, g in df.groupby("seed"):
        sc = S[g.index].mean(0); t = float(np.median(g.thr)); P, R, F1 = prf(y, (sc >= t).astype(int))
        E.append({"auc": safe_auc(y, sc), "ap": safe_ap(y, sc), "F1": 100*F1, "P": 100*P, "R": 100*R})
    E = pd.DataFrame(E)
    row.update({"ens_auc": E.auc.mean(), "ens_auc_sd": _sd(E.auc), "ens_ap": E.ap.mean(),
                "ens_F1": E.F1.mean(), "ens_F1_sd": _sd(E.F1), "ens_P": E.P.mean(), "ens_R": E.R.mean(),
                "ens_F1_oracle_REFERENCE_ONLY": 100*best_threshold(y, S.mean(0))[0]})
    return row


EVONLY = [f"{b}+ev{m}" for b in ("full", "doc_only") for m in ("ablate", "random")]
ALL_SPECS = TIER1 + TIER2 + CONTROLS + EVONLY
rows = [summarize(n) for n in ALL_SPECS]
rows += [summarize(n, "exh") for n in CFG.exh_specs]
SUMMARY = pd.DataFrame([r for r in rows if r])
SUMMARY.to_csv(TAB/"summary_all.csv", index=False)
show = ["spec", "source", "n_models", "auc", "auc_sd", "ens_auc", "ap", "F1", "F1_sd", "P", "R", "ens_F1"]
print(SUMMARY[show].round(3).to_string(index=False))
print(f"\ntrivial: all-positive F1 = {TRIV['all_positive_F1']:.1f} | prevalence-random F1 = "
      f"{TRIV['prevalence_random_F1']:.1f} | chance AUC = 0.500")


def main_table():
    """Replaces the submitted Tables 3/4: trivial rows, published and reproduced baselines,
    whole-image CNN, and DocMIL with both fold-mean and fold-ensemble values."""
    out = [{"model": "Trivial: all forged", "F1": TRIV["all_positive_F1"], "ROC-AUC": .5},
           {"model": "Trivial: prevalence-random", "F1": TRIV["prevalence_random_F1"], "ROC-AUC": .5}]
    out += PAPER_BASELINES + CLASSICAL + [DCT_ROW]
    names = {"whole_image": "Whole-image CNN (same encoder)", "doc_only": "DocMIL, document labels only",
             "full": "DocMIL (full)", "full_mean": "DocMIL (full), mean pooling"}
    for k, lab in names.items():
        for src in ("bag", "exh"):
            r = SUMMARY[(SUMMARY.spec == k) & (SUMMARY.source == src)]
            if r.empty: continue
            r = r.iloc[0]; tag = lab + (" [exhaustive tiling]" if src == "exh" else "")
            out.append({"model": tag, "P": r.P, "R": r.R, "F1": r.F1, "F1_sd": r.F1_sd, "ROC-AUC": r.auc,
                        "AUC_sd": r.auc_sd, "PR-AUC": r.ap, "ens_AUC": r.ens_auc, "ens_F1": r.ens_F1,
                        "params": r.params, "n_models": r.n_models})
    df = pd.DataFrame(out); df.to_csv(TAB/"table_main.csv", index=False)
    print(df.round(3).to_string(index=False)); return df


MAIN = main_table()


In [ ]:
def _strat_boot(y, B, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    return np.concatenate([rng.choice(pos, (B, len(pos))), rng.choice(neg, (B, len(neg)))], 1)


def _jack_groups(n, max_groups=400, seed=0):
    if n <= max_groups: return [np.array([i]) for i in range(n)]
    return np.array_split(np.random.default_rng(seed).permutation(n), max_groups)


def bca(theta, boot, jack, alpha=0.05):
    boot = boot[np.isfinite(boot)]; B = len(boot)
    z0 = stats.norm.ppf(np.clip((boot < theta).mean()+.5*(boot == theta).mean(), 1/(2*B), 1-1/(2*B)))
    jm = jack.mean(); num = ((jm-jack)**3).sum(); den = 6.*(((jm-jack)**2).sum())**1.5
    a = num/den if den > 0 else 0.

    def adj(al):
        z = stats.norm.ppf(al); return stats.norm.cdf(z0+(z0+z)/(1-a*(z0+z)))
    lo, hi = np.quantile(boot, [adj(alpha/2), adj(1-alpha/2)])
    p0 = np.clip((boot < 0).mean()+.5*(boot == 0).mean(), 1/(2*B), 1-1/(2*B))
    w = stats.norm.ppf(p0)-z0
    if 1+a*w <= 0: p = 1/B
    else:
        zal = w/(1+a*w)-z0; p = 2*min(stats.norm.cdf(zal), 1-stats.norm.cdf(zal))
    return float(lo), float(hi), float(min(max(p, 0.), 1.)), float(z0), float(a)


def paired_test(Sa, Sb, y, B=None, seed=0):
    """Δ mean-model AUC, BCa CI and inverted p. Sb=None tests against chance (0.5)."""
    B = B or CFG.n_boot; y = np.asarray(y).astype(int); Sa = np.atleast_2d(Sa)
    Sb = None if Sb is None else np.atleast_2d(Sb)

    def th(ix, yy):
        v = auc_rows(Sa[:, ix], yy).mean()
        return v - (auc_rows(Sb[:, ix], yy).mean() if Sb is not None else .5)
    allix = np.arange(len(y)); theta = th(allix, y)
    I = _strat_boot(y, B, seed); yb = y[I[0]]
    boot = np.array([th(I[b], yb) for b in range(B)])
    jack = []
    for g in _jack_groups(len(y)):
        keep = np.ones(len(y), bool); keep[g] = False; jack.append(th(allix[keep], y[keep]))
    lo, hi, p, z0, a = bca(theta, boot, np.array(jack), CFG.alpha)
    return {"delta": float(theta), "ci_lo": lo, "ci_hi": hi, "p_bca": p}


def mixed_effects(Sa, ra, Sb, rb, y):
    A = {(r["seed"], r["fold"]): v for r, v in zip(ra, auc_rows(Sa, y))}
    Bm = {(r["seed"], r["fold"]): v for r, v in zip(rb, auc_rows(Sb, y))}
    keys = sorted(set(A) & set(Bm))
    if len(keys) < 3: return {"n_pairs": len(keys), "lmm_est": np.nan, "lmm_se": np.nan, "p_lmm": np.nan, "lmm": "n/a"}
    df = pd.DataFrame([{"d": A[k]-Bm[k], "seed": k[0], "fold": k[1]} for k in keys])
    if df.seed.nunique() > 1 and df.fold.nunique() > 1:
        try:
            import statsmodels.formula.api as smf
            res = smf.mixedlm("d ~ 1", df, groups=df["fold"]).fit(reml=True, method="lbfgs")
            return {"n_pairs": len(df), "lmm_est": float(res.params["Intercept"]),
                    "lmm_se": float(res.bse["Intercept"]), "p_lmm": float(res.pvalues["Intercept"]),
                    "lmm": "d ~ 1 + (1|fold)"}
        except Exception as e:
            print("  [lmm fallback]", e)
    t = stats.ttest_1samp(df.d, 0.)
    return {"n_pairs": len(df), "lmm_est": float(df.d.mean()), "lmm_se": float(df.d.std(ddof=1)/math.sqrt(len(df))),
            "p_lmm": float(t.pvalue), "lmm": "one-sample t over fold x seed (not independent)"}


FAMILIES = {
    "supervision": [("full", "doc_only"), ("full", "sampling_only"), ("full", "aux_only"),
                    ("sampling_only", "doc_only"), ("aux_only", "doc_only")],
    "aggregation": [("full", "full_mean"), ("full", "full_max"), ("doc_only", "doc_only_mean"),
                    ("doc_only", "doc_only_max")],
    "baseline":    [("full", "whole_image"), ("doc_only", "whole_image")],
    "transfer":    [("full", "no_pretrain")],
    "controls":    [("doc_only", "doc_only_evid"), ("doc_only_randrm", "doc_only_evid"),
                    ("doc_only", "doc_only_randrm"), ("full", "full+evablate"), ("doc_only", "doc_only+evablate")],
}


def compare_families(families=FAMILIES, source="bag"):
    out = []
    for fam, pairs in families.items():
        rows = []
        for a, b in pairs:
            La, Lb = load_runs(a, source), load_runs(b, source)
            if La is None or Lb is None: continue
            if not np.array_equal(La[3], Lb[3]): print(f"[skip] {a} vs {b}: different test docs"); continue
            y = La[2]; r = {"family": fam, "A": a, "B": b, "n_models_A": len(La[0]), "n_models_B": len(Lb[0]),
                            "auc_A": float(auc_rows(La[1], y).mean()), "auc_B": float(auc_rows(Lb[1], y).mean())}
            r.update(paired_test(La[1], Lb[1], y)); r.update(mixed_effects(La[1], La[0], Lb[1], Lb[0], y))
            rows.append(r)
        if rows:
            p = holm([r["p_bca"] for r in rows])
            for r, q in zip(rows, p): r["p_holm"] = q; r["significant"] = bool(q < CFG.alpha)
            out += rows
    df = pd.DataFrame(out)
    if len(df):
        df["p_bca_fmt"] = df.p_bca.map(fmt_p); df["p_holm_fmt"] = df.p_holm.map(fmt_p); df["p_lmm_fmt"] = df.p_lmm.map(fmt_p)
        df.to_csv(TAB/f"stats_pairs_{source}.csv", index=False)
        print(df[["family", "A", "B", "auc_A", "auc_B", "delta", "ci_lo", "ci_hi", "p_bca_fmt", "p_holm_fmt",
                  "lmm_est", "p_lmm_fmt", "n_pairs"]].round(4).to_string(index=False))
    return df


def chance_tests(names, source="bag"):
    rows = []
    for n in names:
        L_ = load_runs(n, source)
        if L_ is None: continue
        r = {"spec": n, "n_models": len(L_[0]), "auc": float(auc_rows(L_[1], L_[2]).mean())}
        r.update(paired_test(L_[1], None, L_[2])); rows.append(r)
    df = pd.DataFrame(rows)
    if len(df):
        df["p_fmt"] = df.p_bca.map(fmt_p); df.to_csv(TAB/f"stats_vs_chance_{source}.csv", index=False)
        print(df.round(4).to_string(index=False))
    return df


# the eval-only ledger names use '+evablate' / '+evrandom'
print("Pairwise comparisons (bag evaluation):"); STATS = compare_families()
print("\nPairwise comparisons (exhaustive tiling):")
STATS_EXH = compare_families({"exhaustive": [("full", "doc_only")]}, "exh")
print("\nAgainst chance:"); CHANCE = chance_tests(TIER1 + CONTROLS + EVONLY)
print("\nNOTE for the text: the fold x seed models share training data, so their test AUCs are not independent "
      "replicates; inference is on test documents (bootstrap) and on the fold x seed structure (LMM).")


In [ ]:
def loc_one(attn, ph, ov, tdoc, theta):
    lab = (ov >= theta).astype(int); o = np.argsort(tdoc, kind="stable")
    u, st = np.unique(tdoc[o], return_index=True); en = np.r_[st[1:], len(o)]
    nb = np.empty(len(tdoc)); per = []
    for d, a0, a1 in zip(u, st, en):
        ix = o[a0:a1]; nb[ix] = len(ix)
        if DLABEL[d] != 1 or HAS_ANN[d] != 1: continue
        l = lab[ix]
        if l.sum() == 0 or l.sum() == len(l): continue
        a, p = attn[ix], ph[ix]; pf = l.mean(); ac = np.clip(a, 0, None)
        per.append({"doc": int(d), "n_tiles": len(ix), "pos_frac": pf,
                    "auc_attn": auc_rows(a[None], l)[0], "auc_phead": auc_rows(p[None], l)[0],
                    "hit_attn": float(l[np.argmax(a)]), "hit_phead": float(l[np.argmax(p)]),
                    "mass_ratio": float(ac[l == 1].sum()/max(ac.sum(), 1e-12)/pf)})
    fd = (DLABEL[tdoc] == 1)
    pooled = {"pooled_auc_raw": safe_auc(lab, attn), "pooled_auc_bagnorm": safe_auc(lab, attn*nb),
              "pooled_auc_bagnorm_forged_docs": safe_auc(lab[fd], (attn*nb)[fd]),
              "pooled_auc_phead": safe_auc(lab, ph)}
    return pd.DataFrame(per), pooled


def localization(name, source="bag", B=2000):
    L_ = load_runs(name, source)
    if L_ is None: return None, None
    Z = get_cache(CFG) if source == "bag" else None; per_all, pooled = [], []
    for r in L_[0]:
        if source == "bag":
            z = np.load(PRED/f"{r['rid']}.npz")
            if "t_attn" not in z.files: return None, None
            a, p, td, ov = z["t_attn"], z["t_phead"], z["t_tdoc"], Z["ov"][z["t_tid"]]
        else:
            e = load_exh(r["rid"]); a, p, td, ov = e["attn"], e["phead"], e["tdoc"], e["ov"]
        per, po = loc_one(a, p, ov, td, CFG.min_overlap); per["rid"] = r["rid"]
        per_all.append(per); pooled.append(po)
    P = pd.concat(per_all, ignore_index=True); PO = pd.DataFrame(pooled)
    n_fa = int(((DLABEL[TEST_IDX] == 1) & (HAS_ANN[TEST_IDX] == 1)).sum())
    row = {"spec": name, "source": source, "n_models": len(L_[0]), "n_forged_annotated_test": n_fa}
    if P.empty:
        row["n_docs_evaluable"] = 0; return row, P
    cols = ["auc_attn", "auc_phead", "hit_attn", "hit_phead", "pos_frac", "mass_ratio"]
    D = P.groupby("doc")[cols].mean(); row["n_docs_evaluable"] = len(D)
    rng = np.random.default_rng(0); I = rng.integers(0, len(D), (B, len(D)))
    for c in cols:
        v = D[c].values; bs = v[I].mean(1)
        row[c] = float(v.mean()); row[c+"_lo"], row[c+"_hi"] = np.percentile(bs, [2.5, 97.5])
    row["hit_chance"] = row.pop("pos_frac"); row.pop("pos_frac_lo"); row.pop("pos_frac_hi")
    for k in PO.columns:
        row[k] = float(PO[k].mean()); row[k+"_sd"] = _sd(PO[k])
    return row, P


LOC_ROWS = []
for n in ["full", "doc_only", "sampling_only", "aux_only", "doc_only_perm", "full_perm"]:
    for src in ("bag", "exh"):
        if src == "exh" and n not in CFG.exh_specs: continue
        r, _ = localization(n, src)
        if r: LOC_ROWS.append(r)
LOC = pd.DataFrame(LOC_ROWS); LOC.to_csv(TAB/"localization.csv", index=False)
cols = ["spec", "source", "n_docs_evaluable", "n_forged_annotated_test", "auc_attn", "auc_attn_lo", "auc_attn_hi",
        "auc_phead", "hit_attn", "hit_chance", "mass_ratio", "pooled_auc_raw", "pooled_auc_bagnorm"]
print(LOC[[c for c in cols if c in LOC]].round(3).to_string(index=False))


def old_attention_diagnostic(old_dir=OLD_OUT_DIR):
    """Re-reads the ORIGINAL outputs and explains the reported pooled attention AUC."""
    old = Path(old_dir) if old_dir else None
    if old is None or not old.exists():
        print("[old diagnostic] original outputs not attached (OLD_OUT_DIR); skipped"); return None
    try:
        if (old/"cache"/"all.npz").exists():
            zc = np.load(old/"cache"/"all.npz", allow_pickle=True); doc, plab, dl = zc["doc"], zc["plabel"], zc["dlabel"]
        else:
            doc = np.load(old/"cache"/"all_doc.npy"); plab = np.load(old/"cache"/"all_plabel.npy")
            dl = np.load(old/"cache"/"all_dlabel.npy")
    except Exception as e:
        print("[old diagnostic] cannot read old cache:", e); return None
    te = np.arange(len(dl)-len(TEST_IDX), len(dl))
    order = np.argsort(doc, kind="stable"); srt = doc[order]
    tiles = [order[np.searchsorted(srt, d, "left"):np.searchsorted(srt, d, "right")] for d in te]
    if CFG.dataset == "rtm": tiles = [np.sort(t) for t in tiles]
    sizes = np.array([len(t) for t in tiles]); tl = np.concatenate(tiles)
    tdoc = np.repeat(np.arange(len(te)), sizes); lab = plab[tl].astype(int); dlab = dl[te]
    nb = np.repeat(sizes, sizes).astype(float); rows = []
    for f in sorted((old/"preds").glob("full__fold*.npz")):
        if "ensemble" in f.name: continue
        z = np.load(f); a = z["attn"]
        if len(a) != len(tl) or not np.array_equal(z["patch_labels"].astype(int), lab):
            print("[skip] order mismatch", f.name); continue
        per = []; mass = []
        for j in np.where(dlab == 1)[0]:
            m = tdoc == j; l = lab[m]
            if 0 < l.sum() < len(l):
                per.append(auc_rows(a[m][None], l)[0]); mass.append(a[m][l == 1].sum()/l.mean())
        rows.append({"file": f.name, "pooled_auc_raw": safe_auc(lab, a), "pooled_auc_bagnorm": safe_auc(lab, a*nb),
                     "within_doc_auc_mean": float(np.mean(per)) if per else np.nan,
                     "mass_ratio_mean": float(np.mean(mass)) if mass else np.nan, "n_docs": len(per)})
    R = pd.DataFrame(rows)
    bag = pd.DataFrame({"forged": dlab, "bag_size": sizes,
                        "forged_tiles": [int(lab[tdoc == j].sum()) for j in range(len(te))]})
    print("OLD test bags (built with annotations):")
    print(bag.groupby("forged")[["bag_size", "forged_tiles"]].describe().round(1).T.to_string())
    if len(R):
        print("\n", R.round(3).to_string(index=False))
        m = R.mean(numeric_only=True)
        print("\nInterpretation:")
        if bag.groupby("forged").bag_size.mean().diff().abs().iloc[-1] > 1:
            print(f" - forged bags held {bag[bag.forged == 1].bag_size.mean():.1f} tiles vs "
                  f"{bag[bag.forged == 0].bag_size.mean():.1f} for authentic: softmax weights in forged bags are "
                  "smaller by construction, which pushes forged tiles down in a pooled ranking.")
        if m.pooled_auc_bagnorm - m.pooled_auc_raw > .05:
            print(f" - normalising by bag size moves pooled AUC {m.pooled_auc_raw:.3f} -> {m.pooled_auc_bagnorm:.3f}.")
        print(f" - within forged documents, mean per-document AUC = {m.within_doc_auc_mean:.3f} "
              f"(mass ratio {m.mass_ratio_mean:.2f}; 1 = uniform).")
    R.to_csv(TAB/"old_attention_diagnostic.csv", index=False); bag.to_csv(TAB/"old_bag_sizes.csv", index=False)
    return R


OLD_DIAG = old_attention_diagnostic()


In [ ]:
def threshold_study(name, targets=(0.05, 0.10, 0.20, 0.30, 0.50), B=None, seed=0):
    L_ = load_runs(name)
    if L_ is None: return None
    recs, S, y, _ = L_; B = B or (1000 if len(y) < 1000 else 500)
    V = [np.load(PRED/f"{r['rid']}.npz") for r in recs]
    vy = [v["v_ylab"].astype(int) for v in V]; vs = [v["v_score"] for v in V]
    rng = np.random.default_rng(seed); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; n = len(y)
    rows = []
    for pi in sorted(set(list(targets)+[round(float(y.mean()), 4)])):
        def w_for(a):
            pv = max(a.mean(), 1e-9); return np.where(a == 1, pi/pv, (1-pi)/max(1-pv, 1e-9))
        thr = {"fixed_0.5": [.5]*len(recs),
               "val_F1_optimal": [best_threshold(a, b)[1] for a, b in zip(vy, vs)],
               "val_F1_optimal_prevalence_reweighted": [best_threshold(a, b, w_for(a))[1] for a, b in zip(vy, vs)]}
        npos = max(1, int(round(pi*n))); nneg = max(1, n-npos)
        F = {k: [] for k in list(thr)+["oracle_test_REFERENCE_ONLY"]}
        for _ in range(B):
            ix = np.r_[rng.choice(pos, npos), rng.choice(neg, nneg)]; yy = y[ix]; Sx = S[:, ix]
            for k, ts in thr.items():
                F[k].append(np.mean([prf(yy, (Sx[m] >= ts[m]).astype(int))[2] for m in range(len(recs))]))
            F["oracle_test_REFERENCE_ONLY"].append(np.mean([best_threshold(yy, Sx[m])[0] for m in range(len(recs))]))
        for k, v in F.items():
            v = 100*np.asarray(v)
            rows.append({"spec": name, "prevalence": pi, "native": abs(pi-y.mean()) < 1e-3, "method": k,
                         "F1": v.mean(), "F1_lo": np.percentile(v, 2.5), "F1_hi": np.percentile(v, 97.5)})
    return pd.DataFrame(rows)


THR = pd.concat([d for d in (threshold_study(n) for n in ("full", "doc_only")) if d is not None], ignore_index=True) \
    if any(load_runs(n) for n in ("full", "doc_only")) else pd.DataFrame()
if len(THR):
    THR.to_csv(TAB/"threshold_prevalence.csv", index=False)
    print(THR.pivot_table(index=["spec", "prevalence"], columns="method", values="F1").round(1).to_string())
    fig, ax = plt.subplots(1, THR.spec.nunique(), figsize=(5.2*THR.spec.nunique(), 3.8), squeeze=False)
    for a, (sp, g) in zip(ax[0], THR.groupby("spec")):
        for mth, gg in g.groupby("method"):
            ls = ":" if "oracle" in mth else "-"
            a.plot(gg.prevalence, gg.F1, ls, marker="o", ms=3, label=mth)
            a.fill_between(gg.prevalence, gg.F1_lo, gg.F1_hi, alpha=.12)
        a.set(xlabel="Test prevalence (resampled)", ylabel="F1 (forged)", title=sp); a.legend(fontsize=6)
    savefig(fig, f"fig_{CFG.dataset}_threshold_prevalence")


In [ ]:
def per_group(name, groups, auth_groups=None, B=2000, seed=0, tag="type"):
    """Forged documents of each group vs authentic documents (optionally group-matched).
    Grand fold x seed ensemble, median validation threshold. CIs: stratified bootstrap (AUC),
    Wilson (recall). Rows with < 30 forged documents are flagged exploratory (R4.4)."""
    L_ = load_runs(name)
    if L_ is None: return pd.DataFrame()
    recs, S, y, D = L_; s = S.mean(0); thr = float(np.median([r["thr"] for r in recs])); rows = []
    for g, ids in groups.items():
        auth = (y == 0) if auth_groups is None else ((y == 0) & np.isin(D, list(auth_groups.get(g, []))))
        m = auth | ((y == 1) & np.isin(D, list(ids))); yy, ss = y[m], s[m]; npos = int(yy.sum())
        if npos < 2 or (yy == 0).sum() < 2: continue
        I = _strat_boot(yy, B, seed); yb = yy[I[0]]
        bs = np.array([auc_rows(ss[ix][None], yb)[0] for ix in I])
        k = int((ss[yy == 1] >= thr).sum()); lo, hi = wilson(k, npos)
        rows.append({tag: g, "n_forged": npos, "n_authentic": int((yy == 0).sum()),
                     "auc": auc_rows(ss[None], yy)[0], "auc_lo": np.percentile(bs, 2.5), "auc_hi": np.percentile(bs, 97.5),
                     "recall": 100*k/npos, "recall_lo": lo, "recall_hi": hi, "exploratory": npos < 30})
    return pd.DataFrame(rows).sort_values("auc") if rows else pd.DataFrame()


test_types = defaultdict(set)
for d in TEST_IDX:
    for t in ALL_RECS[d]["types"]: test_types[t].add(d)
PER_TYPE = per_group("full", dict(test_types))
if len(PER_TYPE):
    PER_TYPE.to_csv(TAB/"per_type.csv", index=False); print(PER_TYPE.round(3).to_string(index=False))

if CFG.dataset == "fia":   # benign modifications as a false-positive source, with Wilson CIs
    L_ = load_runs("full")
    if L_ is not None:
        recs, S, y, D = L_; s = S.mean(0); thr = float(np.median([r["thr"] for r in recs])); rows = []
        for g, fn in [("authentic, clean", lambda r: not r["hand_ann"] and not r["digital_ann"]),
                      ("authentic, handwriting", lambda r: r["hand_ann"]),
                      ("authentic, digital annotation", lambda r: r["digital_ann"] and not r["hand_ann"])]:
            m = np.array([(not ALL_RECS[d]["forged"]) and bool(fn(ALL_RECS[d])) for d in D])
            n = int(m.sum()); k = int((s[m] >= thr).sum()) if n else 0; lo, hi = wilson(k, n)
            rows.append({"group": g, "n": n, "false_positive_%": 100*k/max(n, 1), "lo": lo, "hi": hi})
        BENIGN = pd.DataFrame(rows); BENIGN.to_csv(TAB/"benign_false_positives.csv", index=False)
        print("\n", BENIGN.round(1).to_string(index=False))


In [ ]:
def _subset_summary(names, seeds, label):
    rows = []
    for n in names:
        r = summarize(n, seeds=seeds)
        if r: r["setting"] = label(n); rows.append(r)
    return pd.DataFrame(rows)


base_seeds = tuple(CFG.seeds_sens)
SENS = _subset_summary([CFG.sens_base]+sens_specs(), base_seeds,
                       lambda n: "base " + str({"scales": CFG.scales, "stride": CFG.stride_frac, "K_b": CFG.bag_size})
                       if n == CFG.sens_base else n.replace("sens_", ""))
if len(SENS):
    SENS.to_csv(TAB/"sensitivity.csv", index=False)
    print(SENS[["setting", "n_models", "auc", "auc_sd", "ap", "F1", "F1_sd"]].round(3).to_string(index=False))
    fig, ax = plt.subplots(figsize=(6.5, .45*len(SENS)+1.6))
    c = ["#2563eb" if s.startswith("base") else "#94a3b8" for s in SENS.setting]
    ax.barh(SENS.setting, SENS.auc, xerr=SENS.auc_sd.fillna(0), color=c, height=.6, error_kw=dict(lw=1, capsize=3))
    ax.axvline(.5, color="k", ls="--", lw=1); ax.set(xlabel="Test ROC-AUC (fold mean ± sd)", title="Sensitivity")
    savefig(fig, f"fig_{CFG.dataset}_sensitivity")

LC_ROWS = []
for f, n in [(1.0, CFG.sens_base)] + list(zip(CFG.lc_fracs, lc_specs())):
    L_ = load_runs(n, seeds=CFG.seeds_lc)
    if L_ is None: continue
    keep = [i for i, r in enumerate(L_[0]) if r["fold"] in CFG.lc_folds]
    if not keep: continue
    a = auc_rows(L_[1][keep], L_[2])
    LC_ROWS.append({"train_fraction": f, "n_models": len(keep), "auc": a.mean(), "auc_sd": _sd(a),
                    "n_train_mean": float(np.mean([L_[0][i].get("n_train", np.nan) for i in keep]))})
LC = pd.DataFrame(LC_ROWS).sort_values("train_fraction") if LC_ROWS else pd.DataFrame()
if len(LC):
    LC.to_csv(TAB/"learning_curve.csv", index=False); print("\n", LC.round(3).to_string(index=False))
    fig, ax = plt.subplots(figsize=(4.8, 3.4))
    ax.errorbar(LC.n_train_mean, LC.auc, yerr=LC.auc_sd.fillna(0), marker="o", capsize=3)
    ax.set(xlabel="Training documents per fold", ylabel="Test ROC-AUC", title="Data scaling")
    savefig(fig, f"fig_{CFG.dataset}_learning_curve")

LOTO_ROWS = []
for t in loto_types():
    L_ = load_runs(f"{CFG.sens_base}@loto_{t}", seeds=CFG.seeds_lc)
    if L_ is None: continue
    recs, S, y, D = L_; keep_f = [r["fold"] for r in recs]
    Lb = load_runs(CFG.sens_base, seeds=CFG.seeds_lc)
    if Lb is None: continue
    bi = [i for i, r in enumerate(Lb[0]) if r["fold"] in keep_f]; m = np.isin(Lb[3], D)
    a_unseen = auc_rows(S, y); a_seen = auc_rows(Lb[1][bi][:, m], Lb[2][m])
    LOTO_ROWS.append({"held_out_technique": t, "n_forged_test": int(y.sum()), "n_models": len(recs),
                      "auc_seen_in_training": a_seen.mean(), "auc_unseen": a_unseen.mean(),
                      "auc_unseen_sd": _sd(a_unseen), "drop": a_seen.mean()-a_unseen.mean()})
LOTO = pd.DataFrame(LOTO_ROWS)
if len(LOTO):
    LOTO.to_csv(TAB/"leave_one_technique_out.csv", index=False); print("\n", LOTO.round(3).to_string(index=False))

EXH_VS_BAG = pd.DataFrame([r for n in CFG.exh_specs for src in ("bag", "exh")
                           for r in [summarize(n, src, seeds=CFG.exh_seeds)] if r])
if len(EXH_VS_BAG):
    EXH_VS_BAG = EXH_VS_BAG[["spec", "source", "n_models", "auc", "auc_sd", "ens_auc", "F1", "ens_F1"]]
EXH_VS_BAG.to_csv(TAB/"exhaustive_vs_bag.csv", index=False); print("\n", EXH_VS_BAG.round(3).to_string(index=False))


In [ ]:
LABELS = {"full": "full", "doc_only": "document labels only", "sampling_only": "region sampling only",
          "aux_only": "aux. instance loss only", "full_mean": "full, mean pooling", "full_max": "full, max pooling",
          "doc_only_mean": "doc-only, mean", "doc_only_max": "doc-only, max", "no_pretrain": "no ImageNet",
          "whole_image": "whole-image CNN", "doc_only_evid": "evidence removed (train+test)",
          "doc_only_randrm": "matched random removal", "doc_only_perm": "label permutation (doc-only)",
          "full_perm": "label permutation (full)", "full+evablate": "full, evidence removed at test",
          "doc_only+evablate": "doc-only, evidence removed at test",
          "full+evrandom": "full, random removal at test", "doc_only+evrandom": "doc-only, random removal at test"}


def fig_roc_pr(names=("full", "doc_only", "full_mean", "whole_image"), name=f"fig_{CFG.dataset}_roc_pr"):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4)); ok = False
    for n in names:
        L_ = load_runs(n)
        if L_ is None: continue
        recs, S, y, _ = L_; s = S.mean(0); a = auc_rows(S, y)
        fpr, tpr, _ = roc_curve(y, s); pr, rc, _ = precision_recall_curve(y, s)
        lab = f"{LABELS.get(n, n)} (fold-mean {a.mean():.3f}±{_sd(a):.3f}; ens {safe_auc(y, s):.3f})"
        ax[0].plot(fpr, tpr, lw=1.5, label=lab); ax[1].plot(rc, pr, lw=1.5, label=f"{LABELS.get(n, n)} (AP {safe_ap(y, s):.3f})")
        ok = True
    if not ok: plt.close(fig); return
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].axhline(PREV, color="gray", ls=":", lw=1, label="prevalence")
    ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (grand ensemble curve)")
    ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall (forged)")
    for a_ in ax: a_.legend(fontsize=6)
    savefig(fig, name)


def fig_bars(names, title, name):
    d = SUMMARY[(SUMMARY.source == "bag") & SUMMARY.spec.isin(names)].copy()
    if d.empty: return
    d["lab"] = d.spec.map(lambda n: LABELS.get(n, n)); d = d.set_index("spec").loc[[n for n in names if n in set(d.spec)]]
    fig, ax = plt.subplots(1, 2, figsize=(10, .5*len(d)+1.8))
    c = ["#2563eb" if n == "full" else ("#16a34a" if n == "doc_only" else "#94a3b8") for n in d.index]
    ax[0].barh(d.lab, d.auc, xerr=d.auc_sd.fillna(0), color=c, height=.6, error_kw=dict(lw=1, capsize=3))
    ax[0].axvline(.5, color="k", ls="--", lw=1, label="chance"); ax[0].set(xlabel="Test ROC-AUC (fold×seed mean ± sd)")
    ax[1].barh(d.lab, d.F1, xerr=d.F1_sd.fillna(0), color=c, height=.6, error_kw=dict(lw=1, capsize=3))
    ax[1].axvline(TRIV["all_positive_F1"], color="k", ls="--", lw=1, label=f"all-positive F1 {TRIV['all_positive_F1']:.1f}")
    ax[1].set(xlabel="Test F1 at validation threshold")
    for a_ in ax: a_.legend(fontsize=7); a_.invert_yaxis()
    fig.suptitle(title, fontsize=9); savefig(fig, name)


def fig_localization(name=f"fig_{CFG.dataset}_localization"):
    if not len(LOC) or "auc_attn" not in LOC: return
    d = LOC.dropna(subset=["auc_attn"]).copy()
    if d.empty: return
    d["lab"] = d.spec.map(lambda n: LABELS.get(n, n)) + " [" + d.source + "]"
    fig, ax = plt.subplots(1, 2, figsize=(10, .45*len(d)+1.8)); yy = np.arange(len(d))
    for off, col, cc in ((-.18, "auc_attn", "#2563eb"), (.18, "auc_phead", "#f59e0b")):
        ax[0].barh(yy+off, d[col], height=.34, color=cc, label="MIL attention" if col == "auc_attn" else "patch head",
                   xerr=[d[col]-d[col+"_lo"], d[col+"_hi"]-d[col]], error_kw=dict(lw=.8, capsize=2))
    ax[0].axvline(.5, color="k", ls="--", lw=1); ax[0].set_yticks(yy); ax[0].set_yticklabels(d.lab, fontsize=7)
    ax[0].set(xlabel="Within-document AUC, forged vs clean tiles"); ax[0].legend(fontsize=7)
    ax[1].barh(yy, d.hit_attn, height=.5, color="#2563eb", label="top-1 attention hit")
    ax[1].scatter(d.hit_chance, yy, color="k", marker="|", s=120, zorder=3, label="chance")
    ax[1].set_yticks(yy); ax[1].set_yticklabels([]); ax[1].set(xlabel="Pointing-game hit rate"); ax[1].legend(fontsize=7)
    for a_ in ax: a_.invert_yaxis()
    savefig(fig, name)


def _outline(ax, rec, shape):
    m = ann_mask(rec, shape[0], shape[1])
    if m is None: return
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in cs:
        x, y, w, h = cv2.boundingRect(c)
        ax.add_patch(plt.Rectangle((x-2, y-2), w+4, h+4, fill=False, edgecolor="lime", lw=1.2))


def fig_attention_maps(spec="full", n_tp=2, n_fp=1, n_fn=1, name=f"fig_{CFG.dataset}_attention_maps"):
    """Whole-document MIL attention (one softmax over every tile, exhaustive tiling) and the
    patch-head probability, for TP / FP / FN examples. Attention is shown relative to uniform
    (1 = 1/N) with a per-document percentile stretch and a colorbar in those units."""
    L_ = load_runs(spec)
    if L_ is None: return
    recs, S, y, D = L_; s = S.mean(0); thr = float(np.median([r["thr"] for r in recs]))
    rid = next((r["rid"] for r in recs if load_exh(r["rid"]) is not None), None)
    if rid is None: print("no exhaustive outputs yet"); return
    e = load_exh(rid); pos = {d: i for i, d in enumerate(e["doc"])}; dpos = {d: i for i, d in enumerate(D)}
    dec = (s >= thr).astype(int)
    pick = ([(D[i], "True positive") for i in np.where((y == 1) & (dec == 1))[0] if HAS_ANN[D[i]]][:n_tp] +
            [(D[i], "False positive") for i in np.where((y == 0) & (dec == 1))[0]][:n_fp] +
            [(D[i], "False negative") for i in np.where((y == 1) & (dec == 0))[0] if HAS_ANN[D[i]]][:n_fn])
    if not pick: print("no examples"); return
    colors = {"True positive": "#16a34a", "False positive": "#ea580c", "False negative": "#dc2626"}
    fig, ax = plt.subplots(len(pick), 4, figsize=(15, 4.6*len(pick)), squeeze=False,
                           gridspec_kw={"wspace": .25, "hspace": .35})
    plt.rcParams["figure.autolayout"] = False

    def paint(vals, tyx, H, W):
        Hp, Wp = max(H, max(CFG.scales)), max(W, max(CFG.scales))
        acc = np.zeros((Hp, Wp), np.float32); cnt = np.zeros((Hp, Wp), np.float32)
        for (yy, xx, ps), v in zip(tyx, vals):
            acc[yy:yy+ps, xx:xx+ps] += v; cnt[yy:yy+ps, xx:xx+ps] += 1
        return (acc/np.maximum(cnt, 1))[:H, :W]

    for row, (d, kind) in enumerate(pick):
        rec = ALL_RECS[d]; i = pos[d]; a0, a1 = e["off"][i], e["off"][i+1]
        im = load_rgb(rec["path"]); H, W = im.shape[:2]; tyx = e["tyx"][a0:a1]
        rel = e["attn"][a0:a1]*(a1-a0)                       # 1.0 = uniform attention
        att = paint(rel, tyx, H, W); ph = paint(e["phead"][a0:a1], tyx, H, W)
        lo, hi = np.percentile(att, [2, 99.5]); hi = max(hi, lo+1e-6)
        lab = (e["ov"][a0:a1] >= CFG.min_overlap)
        top_hit = "yes" if lab.any() and lab[np.argmax(rel)] else ("no" if lab.any() else "n/a")

        ax[row, 0].imshow(im); _outline(ax[row, 0], rec, im.shape)
        ax[row, 0].set_xlabel(f"{rec['name']}\np(forged) = {s[dpos[d]]:.2f}  (thr {thr:.2f})\n"
                              f"{a1-a0} tiles | max attention {rel.max():.2f}× uniform | top tile on forgery: {top_hit}",
                              fontsize=7)
        ax[row, 0].set_ylabel(kind, fontsize=10, color=colors[kind], fontweight="bold")

        m1 = ax[row, 1].imshow(att, cmap="magma", vmin=lo, vmax=hi)
        cb = fig.colorbar(m1, ax=ax[row, 1], fraction=.046, pad=.02); cb.ax.tick_params(labelsize=6)
        cb.set_label("attention / uniform", fontsize=6)

        m2 = ax[row, 2].imshow(ph, cmap="magma", vmin=0, vmax=1)
        cb = fig.colorbar(m2, ax=ax[row, 2], fraction=.046, pad=.02); cb.ax.tick_params(labelsize=6)
        cb.set_label("patch-head p(forged)", fontsize=6)

        ax[row, 3].imshow(im); ax[row, 3].imshow(att, cmap="magma", alpha=.55, vmin=lo, vmax=hi)
        _outline(ax[row, 3], rec, im.shape)
        for a_ in ax[row]:
            a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)
            for sp_ in a_.spines.values(): sp_.set_visible(False)

    for c, t in enumerate(["Document (annotation in green)", "MIL attention (whole-document softmax)",
                           "Patch-head probability", "Attention overlay"]):
        ax[0, c].set_title(t, fontsize=9, pad=8)
    fig.suptitle(f"{DS if 'DS' in globals() else CFG.dataset.upper()}: {LABELS.get(spec, spec)}, model {rid}",
                 fontsize=9, y=1.0)
    savefig(fig, name)
    plt.rcParams["figure.autolayout"] = True


fig_roc_pr()
fig_bars(["full", "sampling_only", "aux_only", "doc_only", "whole_image"], "Supervision ablation and whole-image baseline",
         f"fig_{CFG.dataset}_supervision")
fig_bars(["full", "full_mean", "full_max", "doc_only", "doc_only_mean", "doc_only_max", "no_pretrain"],
         "Aggregation and transfer", f"fig_{CFG.dataset}_aggregation")
fig_bars(["doc_only", "doc_only_randrm", "doc_only_evid", "doc_only_perm", "full_perm", "full+evablate",
          "doc_only+evablate"], "Controls", f"fig_{CFG.dataset}_controls")
fig_localization()
fig_attention_maps()


In [ ]:
def bench_cpu(model, inputs, label, runs=30):
    m = model.eval().cpu(); torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
    with torch.no_grad():
        for _ in range(5): m(*inputs)
        t = []
        for _ in range(runs):
            s = time.perf_counter(); m(*inputs); t.append((time.perf_counter()-s)*1000)
    t = np.array(t); out = {"model": label, "params": nparams(m), "ms_per_document": float(t.mean()),
                            "p95_ms": float(np.percentile(t, 95)), "threads": torch.get_num_threads()}
    try:
        from thop import profile
        macs, _ = profile(m, inputs=inputs, verbose=False); out["gflops_per_document"] = float(2*macs/1e9)
    except Exception:
        out["gflops_per_document"] = float("nan")
    return out


b = CFG.scales[0]
EFF = pd.DataFrame([bench_cpu(DocMIL(CFG, k), (torch.rand(CFG.k_max, 3, b, b), torch.zeros(CFG.k_max, dtype=torch.long),
                                               1, torch.ones(CFG.k_max)), f"DocMIL ({k}), K_max={CFG.k_max} tiles")
                    for k in MILS] +
                   [bench_cpu(WholeImageCNN(CFG), (torch.rand(1, 3, CFG.whole_res, CFG.whole_res),),
                              f"Whole-image CNN ({CFG.whole_res}px)")])
EFF.to_csv(TAB/"efficiency.csv", index=False); print(EFF.round(3).to_string(index=False))
_v = AUDIT.dropna(subset=["H", "W"]).head(200) if "H" in AUDIT else pd.DataFrame()
if len(_v):
    _mx = max(CFG.scales)
    _nt = [sum(len(tile_grid(max(int(h), _mx), max(int(w), _mx), ps, CFG.stride_frac)) for ps in CFG.scales)
           for h, w in zip(_v.H, _v.W)]
    print(f"Exhaustive tiling: median {int(np.median(_nt))} tiles per document over all scales (first 200 docs).")


In [ ]:
# Run once (needs Kaggle internet). Tesseract with English + Simplified Chinese.
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-chi-sim > /dev/null
!pip install -q pytesseract transformers


In [ ]:
RUN_OCR = False
OCR_LANG = "eng" if CFG.dataset == "fia" else "eng+chi_sim"
OCR_DIR = OUT/"ocr"; OCR_DIR.mkdir(exist_ok=True)


def ocr_doc(r, max_side=2200):
    f = OCR_DIR/f"{r['rid']}.json"
    if f.exists(): return json.loads(f.read_text())
    import pytesseract
    im = Image.open(r["path"]).convert("RGB"); W, H = im.size; s = min(1., max_side/max(W, H))
    if s < 1: im = im.resize((max(1, int(W*s)), max(1, int(H*s))), Image.LANCZOS)
    w2, h2 = im.size
    d = pytesseract.image_to_data(im, lang=OCR_LANG, output_type=pytesseract.Output.DICT)
    words, boxes, lines = [], [], defaultdict(list)
    for t, x, y, w, h, c, b, p, l in zip(d["text"], d["left"], d["top"], d["width"], d["height"], d["conf"],
                                         d["block_num"], d["par_num"], d["line_num"]):
        t = str(t).strip()
        try: c = float(c)
        except Exception: c = -1
        if not t or c < 0: continue
        words.append(t); lines[(b, p, l)].append(t)
        boxes.append([int(np.clip(1000*x/w2, 0, 1000)), int(np.clip(1000*y/h2, 0, 1000)),
                      int(np.clip(1000*(x+w)/w2, 0, 1000)), int(np.clip(1000*(y+h)/h2, 0, 1000))])
    out = {"words": words, "boxes": boxes, "text": "\n".join(" ".join(v) for v in lines.values())}
    f.write_text(json.dumps(out, ensure_ascii=False)); return out


def run_ocr(docs):
    eta = ETA(len(docs), skip=0)
    with ThreadPoolExecutor(max(1, CFG.num_workers)) as ex:
        for c0 in range(0, len(docs), 32):
            list(ex.map(lambda d: ocr_doc(ALL_RECS[d]), docs[c0:c0+32])); eta.step(min(32, len(docs)-c0))
            if (c0//32) % 10 == 0: print("  OCR:", eta.bar())


AMT = re.compile(r"(?<![\d.,])(\d{1,6}[.,]\d{2})(?![\d])")
DATE = re.compile(r"\b(\d{1,2})[/.-](\d{1,2})[/.-](\d{2,4})\b")
RATES = np.array([.05, .06, .07, .08, .10, .15, .20])


def doc_text(r):
    if r.get("txt"):
        return Path(r["txt"]).read_text(encoding="utf-8", errors="replace")
    return ocr_doc(r)["text"]


def semantic_features(text):
    """Receipt-arithmetic and date consistency features (missing value -> -1)."""
    lines = [l.upper() for l in text.splitlines() if l.strip()]
    amt = [(l, [float(a.replace(",", ".")) for a in AMT.findall(l)]) for l in lines]
    allv = [a for _, v in amt for a in v]

    def last(pat, excl=None):
        c = [v for l, v in amt if v and re.search(pat, l) and not (excl and re.search(excl, l))]
        return c[-1][-1] if c else np.nan
    total = last(r"\bTOTAL\b|\bAMOUNT DUE\b|\bGRAND\b", r"SUB|QTY|ITEM|EXCL|INCL\s+GST\s+0")
    sub = last(r"SUB\s*-?\s*TOTAL|EXCL|NET\s+AMOUNT"); tax = last(r"\bGST\b|\bTAX\b|\bSST\b|\bVAT\b|\bTVA\b")
    cash = last(r"\bCASH\b|\bPAID\b|\bTENDER"); change = last(r"\bCHANGE\b"); rnd = last(r"ROUND")
    key = r"TOTAL|SUB|GST|TAX|SST|VAT|CASH|PAID|TENDER|CHANGE|ROUND|DISC|AMOUNT|BALANCE|QTY"
    items = [v[-1] for l, v in amt if v and not re.search(key, l)]
    f = {"n_amounts": len(allv), "n_lines": len(lines), "has_total": float(np.isfinite(total))}
    f["total_is_max"] = float(np.isfinite(total) and bool(allv) and abs(total-max(allv)) < 1e-6) if allv else -1.
    ok = lambda *v: all(np.isfinite(x) for x in v)
    f["sub_tax_resid"] = abs(sub+tax-total)/max(total, 1) if ok(sub, tax, total) else -1.
    f["tax_rate_resid"] = float(np.min(np.abs(tax-RATES*sub))/max(sub, 1)) if ok(tax, sub) and sub > 0 else -1.
    f["cash_change_resid"] = abs(cash-change-total)/max(total, 1) if ok(cash, change, total) else -1.
    ref = sub if np.isfinite(sub) else total
    f["items_resid"] = abs(sum(items)-ref)/max(ref, 1) if items and np.isfinite(ref) else -1.
    f["rounding_abs"] = abs(rnd) if np.isfinite(rnd) else -1.
    tot_lines = [v for l, v in amt if v and re.search(r"\bTOTAL\b", l) and not re.search(r"SUB|QTY", l)]
    f["total_disagree"] = float(len({round(v[-1], 2) for v in tot_lines}) > 1) if tot_lines else -1.
    ds = DATE.findall(" ".join(lines)); f["n_dates"] = len(ds)
    f["date_invalid"] = float(any(int(dd) > 31 or int(mm) > 12 or int(dd) == 0 or int(mm) == 0 for dd, mm, _ in ds)) if ds else -1.
    f["dates_disagree"] = float(len(set(ds)) > 1) if ds else -1.
    return f


def layoutlm_features(docs):
    fp = OCR_DIR/"layoutlmv3_feats.npz"
    X = np.zeros((len(ALL_RECS), 1536), np.float32); done = np.zeros(len(ALL_RECS), bool)
    if fp.exists():
        z = np.load(fp); X, done = z["X"], z["done"]
    from transformers import LayoutLMv3Processor, LayoutLMv3Model
    proc = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
    mdl = LayoutLMv3Model.from_pretrained("microsoft/layoutlmv3-base").to(DEVICE).eval()
    todo = [d for d in docs if not done[d]]; eta = ETA(len(todo), skip=0)
    for i, d in enumerate(todo):
        o = ocr_doc(ALL_RECS[d]); words = o["words"] or ["[EMPTY]"]; boxes = o["boxes"] or [[0, 0, 0, 0]]
        im = Image.open(ALL_RECS[d]["path"]).convert("RGB")
        enc = proc(im, words, boxes=boxes, truncation=True, max_length=512, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad(), torch.autocast(DEVICE.type, enabled=AMP_ON):
            h = mdl(**enc).last_hidden_state.float()[0]
        X[d] = torch.cat([h[0], h.mean(0)]).cpu().numpy(); done[d] = True; eta.step()
        if (i+1) % 200 == 0 or i+1 == len(todo):
            np.savez(fp, X=X, done=done); print("  LayoutLMv3:", eta.bar())
    del mdl; gc.collect(); return X


def docmil_oof(name="full"):
    """Out-of-fold pool scores (each pool doc scored by the fold model that did not train on it),
    averaged over seeds; test = grand fold x seed ensemble."""
    L_ = load_runs(name)
    if L_ is None: return None, None
    acc = np.zeros(len(ALL_RECS)); cnt = np.zeros(len(ALL_RECS))
    for r in L_[0]:
        z = np.load(PRED/f"{r['rid']}.npz"); acc[z["v_doc"]] += z["v_score"]; cnt[z["v_doc"]] += 1
    oof = np.where(cnt > 0, acc/np.maximum(cnt, 1), np.nan); test = np.full(len(ALL_RECS), np.nan)
    test[L_[3]] = L_[1].mean(0)
    return oof, test


def fuse(mil_name, sk_name):
    o1, t1 = docmil_oof(mil_name)
    if o1 is None or not (PRED/f"sk_{sk_name}.npz").exists(): return None
    sk = np.load(PRED/f"sk_{sk_name}.npz"); o2, t2 = sk["oof"], sk["test"]
    lg = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6)/(1-np.clip(p, 1e-6, 1-1e-6)))
    Xp = np.c_[lg(o1[POOL_IDX]), lg(o2[POOL_IDX])]; Xt = np.c_[lg(t1[TEST_IDX]), lg(t2)]
    yp = DLABEL[POOL_IDX]; y = DLABEL[TEST_IDX].astype(int)
    from sklearn.model_selection import cross_val_predict
    mk = lambda: LogisticRegression(class_weight="balanced", max_iter=1000)
    oof = cross_val_predict(mk(), Xp, yp, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    thr = best_threshold(yp, oof)[1]; s = mk().fit(Xp, yp).predict_proba(Xt)[:, 1]
    P, R, F1 = prf(y, (s >= thr).astype(int))
    row = {"model": f"fusion: {mil_name} + {sk_name}", "P": 100*P, "R": 100*R, "F1": 100*F1,
           "ROC-AUC": safe_auc(y, s), "PR-AUC": safe_ap(y, s)}
    row.update({f"vs_docmil_{k}": v for k, v in paired_test(s[None], t1[TEST_IDX][None], y).items()})
    return row


def script_of(text):
    cjk = len(re.findall(r"[\u3400-\u9fff]", text)); lat = len(re.findall(r"[A-Za-z]", text))
    if cjk+lat < 10: return "little text"
    return "Chinese script" if cjk/(cjk+lat) > .3 else "Latin script"


if RUN_OCR:
    need = list(POOL_IDX)+list(TEST_IDX)
    run_ocr(need)                     # LayoutLMv3 needs word boxes even where transcripts exist
    SEM = pd.DataFrame([semantic_features(doc_text(ALL_RECS[d])) for d in range(len(ALL_RECS))]).fillna(-1.)
    SEM.to_csv(TAB/"semantic_features.csv", index=False)
    OCR_ROWS = [cv_sklearn(SEM.values.astype(np.float32), lambda: make_pipeline(
        StandardScaler(), LogisticRegression(C=.5, max_iter=3000, class_weight="balanced")), "semantic_consistency_lr")]
    XL = layoutlm_features(need)
    OCR_ROWS.append(cv_sklearn(XL, lambda: make_pipeline(StandardScaler(), LogisticRegression(
        C=.05, max_iter=5000, class_weight="balanced")), "layoutlmv3_frozen_probe"))
    for sk in ("semantic_consistency_lr", "layoutlmv3_frozen_probe", "dct_first_digit_lr"):
        r = fuse("full", sk)
        if r: OCR_ROWS.append(r)
    OCR_TAB = pd.DataFrame(OCR_ROWS); OCR_TAB.to_csv(TAB/"ocr_semantic_fusion.csv", index=False)
    print(OCR_TAB.round(3).to_string(index=False))
    scr = {d: script_of(ocr_doc(ALL_RECS[d])["text"]) for d in TEST_IDX}
    grp_f, grp_a = defaultdict(set), defaultdict(set)
    for d, g in scr.items(): (grp_f if DLABEL[d] == 1 else grp_a)[g].add(d)
    LANG = per_group("full", dict(grp_f), dict(grp_a), tag="script")
    if len(LANG): LANG.to_csv(TAB/"per_script.csv", index=False); print("\n", LANG.round(3).to_string(index=False))


# ---- ADCD-Net zero-shot hook (R4.3) ----------------------------------------------------------
# Clone https://github.com/KahimWong/ADCD-Net, download ADCDNet.pth + docres.pkl per its README,
# and implement adcd_predict_map(path) -> HxW forgery-probability map. Document score = max of the map.
adcd_predict_map = None


def zero_shot_external(predict_map, name="adcd_net_zero_shot", agg=np.max):
    s = np.zeros(len(TEST_IDX), np.float32); eta = ETA(len(TEST_IDX), skip=0)
    for i, d in enumerate(TEST_IDX):
        s[i] = float(agg(predict_map(ALL_RECS[d]["path"]))); eta.step()
        if (i+1) % 200 == 0: print("  ", eta.bar())
    y = DLABEL[TEST_IDX].astype(int); f1o, _ = best_threshold(y, s)
    row = {"model": name, "ROC-AUC": safe_auc(y, s), "PR-AUC": safe_ap(y, s), "F1_at_0.5": 100*prf(y, (s >= .5).astype(int))[2],
           "F1_oracle_REFERENCE_ONLY": 100*f1o}
    np.save(PRED/f"{name}_test_scores.npy", s)
    L_ = load_runs("full")
    if L_ is not None: row.update({f"docmil_minus_{k}": v for k, v in paired_test(L_[1], s[None], y).items()})
    pd.DataFrame([row]).to_csv(TAB/f"{name}.csv", index=False); print(row); return row


if adcd_predict_map is not None:
    zero_shot_external(adcd_predict_map)


In [ ]:
@torch.no_grad()
def cross_corpus(names=("full", "doc_only"), other_dir=OTHER_OUT_DIR):
    if not other_dir or not Path(other_dir).exists():
        print("[cross-corpus] OTHER_OUT_DIR not set; skipped"); return None
    cands = []
    for m in Path(other_dir).glob("cache/*/meta.json"):
        meta = json.loads(m.read_text())
        if meta.get("complete") and meta.get("base") == CFG.scales[0] and tuple(meta["cfg"]["scales"]) == tuple(CFG.scales):
            cands.append(m.parent)
    if not cands: print("[cross-corpus] no compatible cache in", other_dir); return None
    Zo = load_cache(cands[0]); mo = Zo["meta"]
    te = np.arange(mo["n_pool"], mo["n_pool"]+mo["n_test"]); y = Zo["dlabel"][te].astype(int)
    k_eval = min(CFG.k_max, int(mo.get("k_pool", CFG.k_max)))
    rows = []; L = ledger_all()
    for n in names:
        sp = get_spec(n)
        for r in [r for r in L.values() if r["name"] == n]:
            ck = CKPT/f"{r['rid']}.pt"
            if not ck.exists(): continue
            st = torch.load(ck, map_location="cpu", weights_only=False)
            model = DocMIL(CFG, sp["mil"], sp["pretrained"]).to(DEVICE); model.load_state_dict(st["model"])
            t = predict_mil(model.to(memory_format=torch.channels_last),
                            mil_loader(Zo, te, CFG, False, r["seed"], 0, k_eval=k_eval), CFG, tta=CFG.tta)
            P, R, F1 = prf(t["ytrue"], (t["score"] >= r["thr"]).astype(int))
            rows.append({"spec": n, "rid": r["rid"], "target": mo["dataset"], "auc": safe_auc(t["ytrue"], t["score"]),
                         "ap": safe_ap(t["ytrue"], t["score"]), "F1_at_source_thr": 100*F1})
            del model; gc.collect()
    df = pd.DataFrame(rows)
    if len(df):
        agg = df.groupby("spec").agg(n_models=("auc", "size"), auc=("auc", "mean"), auc_sd=("auc", "std"),
                                     ap=("ap", "mean"), F1=("F1_at_source_thr", "mean")).reset_index()
        agg["source"] = CFG.dataset; agg["target"] = mo["dataset"]
        agg.to_csv(TAB/f"cross_corpus_{CFG.dataset}_to_{mo['dataset']}.csv", index=False)
        print(agg.round(3).to_string(index=False)); return agg


CROSS = cross_corpus()


In [ ]:
def df_to_latex(df, cols, caption, label, path, nd=3):
    cols = [c for c in cols if c in df.columns]
    esc = lambda s: str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&").replace("†", r"$^\dagger$")
    lines = [r"\begin{table}[t]", r"\centering", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{l"+"r"*(len(cols)-1)+"}", r"\toprule", " & ".join(esc(c) for c in cols)+r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r.get(c)
            if v is None or (isinstance(v, float) and not np.isfinite(v)): cells.append("--")
            elif c == "params": cells.append(f"{int(v)/1e6:.2f}M")
            elif isinstance(v, (float, np.floating)): cells.append(f"{v:.{nd}f}" if abs(v) < 1.5 else f"{v:.1f}")
            else: cells.append(esc(v))
        lines.append(" & ".join(cells)+r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    Path(path).write_text("\n".join(lines)); return "\n".join(lines)


DS = {"fia": "FindItAgain", "rtm": "RTM"}[CFG.dataset]
if len(MAIN):
    df_to_latex(MAIN, ["model", "P", "R", "F1", "F1_sd", "ROC-AUC", "AUC_sd", "PR-AUC", "ens_AUC", "ens_F1", "params"],
                f"Document-level detection on {DS}. DocMIL rows: mean $\\pm$ sd over fold$\\times$seed models and "
                f"per-seed fold ensembles (ens). Thresholds from validation folds only. Trivial all-forged F1 = "
                f"{TRIV['all_positive_F1']:.1f}; prevalence-random F1 = {TRIV['prevalence_random_F1']:.1f}.",
                f"tab:{CFG.dataset}_main", TAB/"table_main.tex")
abl = SUMMARY[(SUMMARY.source == "bag") & SUMMARY.spec.isin(TIER1+TIER2+CONTROLS+EVONLY)].copy()
if len(abl):
    abl.insert(0, "variant", abl.spec.map(lambda n: LABELS.get(n, n)))
    df_to_latex(abl, ["variant", "n_models", "auc", "auc_sd", "ens_auc", "ap", "F1", "F1_sd"],
                f"Supervision, aggregation, baseline and control variants on {DS} (annotation-blind evaluation bags).",
                f"tab:{CFG.dataset}_ablation", TAB/"table_ablation.tex")
if len(STATS):
    df_to_latex(STATS.assign(pair=STATS.A+" vs "+STATS.B),
                ["pair", "delta", "ci_lo", "ci_hi", "p_bca_fmt", "p_holm_fmt", "lmm_est", "p_lmm_fmt"],
                f"Paired fold-ensemble analysis on {DS}: $\\Delta$ mean-model ROC-AUC, BCa 95\\% CI, inverted BCa p, "
                "Holm-adjusted within family, and a fold random-intercept mixed model on per-model AUC differences.",
                f"tab:{CFG.dataset}_stats", TAB/"table_stats.tex", nd=4)
if len(LOC):
    df_to_latex(LOC.assign(v=LOC.spec.map(lambda n: LABELS.get(n, n))+" ["+LOC.source+"]"),
                ["v", "n_docs_evaluable", "auc_attn", "auc_phead", "hit_attn", "hit_chance", "mass_ratio",
                 "pooled_auc_raw", "pooled_auc_bagnorm"],
                f"Attention-annotation agreement inside forged test documents on {DS}.",
                f"tab:{CFG.dataset}_loc", TAB/"table_localization.tex")


def _mac(*parts):
    w = {"0": "Zero", "1": "One", "2": "Two", "3": "Three", "4": "Four", "5": "Five", "6": "Six", "7": "Seven",
         "8": "Eight", "9": "Nine"}
    s = "".join(p.replace("+", " ").replace("_", " ").title().replace(" ", "") for p in parts)
    return "".join(w.get(ch, ch) for ch in s if ch.isalnum())


macros = []
for _, r in SUMMARY.iterrows():
    for m, nd in (("auc", 3), ("auc_sd", 3), ("ens_auc", 3), ("ap", 3), ("F1", 1), ("F1_sd", 1), ("ens_F1", 1)):
        v = r.get(m)
        if v is not None and np.isfinite(v):
            macros.append(rf"\newcommand{{\{_mac(DS, r.spec, r.source, m)}}}{{{v:.{nd}f}}}")
for k, v in TRIV.items(): macros.append(rf"\newcommand{{\{_mac(DS, 'triv', k)}}}{{{v:.1f}}}")
if len(STATS):
    for _, r in STATS.iterrows():
        b = _mac(DS, r.A, "vs", r.B)
        macros += [rf"\newcommand{{\{b}Delta}}{{{r.delta:.3f}}}", rf"\newcommand{{\{b}CI}}{{[{r.ci_lo:.3f}, {r.ci_hi:.3f}]}}",
                   rf"\newcommand{{\{b}P}}{{{r.p_holm_fmt}}}"]
(TAB/"numbers.tex").write_text("\n".join(macros)); print(len(macros), "macros -> tables/numbers.tex")

# ---- reproducibility manifest (R3.11, R4.3) ----
from torchvision.models import MobileNet_V3_Small_Weights
enc = PretrainedEncoder(CFG.freeze_until)
manifest = {
    "dataset": CFG.dataset, "root": CFG.root, "config": asdict(CFG), "k_pool": CFG.k_pool,
    "presets_diff": {k: {"fia": PRESETS["fia"].get(k, getattr(Config(), k)), "rtm": PRESETS["rtm"].get(k, getattr(Config(), k))}
                     for k in sorted(set(PRESETS["rtm"]) | set(PRESETS["fia"])) if k not in ("root", "out_dir", "dataset", "cache_dir")},
    "retention_rule": "uniform random draw without replacement from the full multi-scale grid, balanced and "
                      "interleaved across scales, RNG seeded by (cache_seed, document index); evaluation bag = "
                      "first K_max; training bag = K_b uniform from first K_train each epoch; annotation-positive "
                      "extras cached for training-pool documents only; annotations never consulted for evaluation bags",
    "theta_min_px2": CFG.min_overlap, "evidence_ablation_rule": "remove every tile with overlap > 0 px^2",
    "frozen_blocks": [f"features[{i}] ({type(enc.f[i]).__name__})" for i in range(CFG.freeze_until)],
    "imagenet_weights": {"enum": "torchvision MobileNet_V3_Small_Weights.IMAGENET1K_V1",
                         "url": MobileNet_V3_Small_Weights.IMAGENET1K_V1.url},
    "attention_normalisation": "softmax over the tiles of one bag (whole document under exhaustive tiling)",
    "tta": "mean of sigmoid outputs over 4 views {identity, h-flip, v-flip, hv-flip}; used for validation "
           "threshold selection and test; not used for epoch selection",
    "model_selection": "best validation ROC-AUC epoch; threshold = F1-optimal on validation (TTA) of that fold",
    "seeds": {"tier1": CFG.seeds, "tier2": CFG.seeds_t2, "controls": CFG.seeds_ctrl, "sensitivity": CFG.seeds_sens,
              "learning_curve/LOTO": CFG.seeds_lc, "exhaustive": CFG.exh_seeds, "fold_split": CFG.split_seed,
              "cache": CFG.cache_seed},
    "counts": COUNTS.to_dict("records"), "listing": LISTING.to_dict("records"),
    "cache": {k: v for k, v in Z0["meta"].items() if k != "cfg"}, "cache_report": CACHE_REPORT.to_dict("records"),
    "splits_file": "splits.json", "split_files_md5": SPLITS["split_files_md5"],
    "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "torchvision": torchvision.__version__,
                 "numpy": np.__version__, "sklearn": sklearn.__version__, "opencv": cv2.__version__,
                 "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None},
    "hardware": {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                 "cpu_count": os.cpu_count(), "platform": platform.platform()},
    "nondeterminism": "cudnn.benchmark=True and TF32 enabled; repeated runs with the same seed can differ slightly",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
(OUT/"manifest.json").write_text(json.dumps(manifest, indent=1, default=_json_default))
os.system(f"pip freeze > {OUT/'requirements_freeze.txt'}")
pd.DataFrame([{"hyperparameter": k, **v} for k, v in manifest["presets_diff"].items()]).to_csv(TAB/"hyperparameters_by_corpus.csv", index=False)
print(pd.read_csv(TAB/"hyperparameters_by_corpus.csv").to_string(index=False))


def _get(df, cond, col):
    try: return df[cond][col].iloc[0]
    except Exception: return float("nan")


S_ = SUMMARY[SUMMARY.source == "bag"]
print(f"\n================ REVIEWER CHECKLIST ({DS}) ================")
print(f"R2.1  doc-only AUC {_get(S_, S_.spec == 'doc_only', 'auc'):.3f} | sampling-only {_get(S_, S_.spec == 'sampling_only', 'auc'):.3f}"
      f" | aux-only {_get(S_, S_.spec == 'aux_only', 'auc'):.3f} | full {_get(S_, S_.spec == 'full', 'auc'):.3f}  -> table_ablation, stats_pairs_bag.csv")
print(f"R2.3/R3.7 whole-image CNN AUC {_get(S_, S_.spec == 'whole_image', 'auc'):.3f}")
print("R2.4/R3.10 attention vs mean: see stats_pairs_bag.csv family 'aggregation'")
print(f"R3.1  exhaustive tiling: exhaustive_vs_bag.csv | eval bags with a forged tile: "
      f"{CACHE_REPORT[CACHE_REPORT.part == 'test']['eval_bags_with_forged_tile_%'].iloc[0]:.1f}% of forged test docs")
print(f"R3.2  evidence ablation AUC {_get(S_, S_.spec == 'doc_only_evid', 'auc'):.3f} vs matched random removal "
      f"{_get(S_, S_.spec == 'doc_only_randrm', 'auc'):.3f}; label permutation {_get(S_, S_.spec == 'doc_only_perm', 'auc'):.3f}"
      f" / {_get(S_, S_.spec == 'full_perm', 'auc'):.3f}; localization.csv; old_attention_diagnostic.csv")
print("R3.3/R4.1 BCa + inverted p + LMM: stats_pairs_bag.csv; fold vs ensemble: summary_all.csv")
print("R3.5  counts.csv, listing.csv (listed vs usable rows), cache_report.csv, format_audit.csv")
print("R3.8  format_audit.csv, dct_baseline.csv")
print("R3.11/R4.3 manifest.json, splits.json, requirements_freeze.txt, hyperparameters_by_corpus.csv")
print("R4.3  sensitivity.csv | R4.4 per_type.csv (exploratory flags) | R4.5 threshold_prevalence.csv")
print("R1    leave_one_technique_out.csv, learning_curve.csv, cross_corpus_*.csv, ocr_semantic_fusion.csv, per_script.csv")


In [ ]:
# ===== Free disk space in /kaggle/working without losing anything that needs retraining =====
import shutil

DRY_RUN = False                 # first run: report only. Set False to apply.
MOVE_CACHE_TO_TMP = True       # move tile/whole-image caches to /tmp (deterministic, rebuilt identically)
DROP_UNUSED_CKPTS = False      # delete best .pt of finished runs whose predictions are saved and that
                               # nothing downstream reloads (keeps full/doc_only/exh specs). Optional.
TMP_CACHE = Path(f"/tmp/docmil_{CFG.dataset}_cache")


def du(p):
    p = Path(p)
    if not p.exists(): return 0
    if p.is_file(): return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


def gb(b): return f"{b/1e9:.2f} GB"


def rm(p, why):
    s = du(p); print(f"  {'[dry] ' if DRY_RUN else ''}remove {gb(s):>9}  {p}  ({why})")
    if not DRY_RUN:
        shutil.rmtree(p, ignore_errors=True) if Path(p).is_dir() else Path(p).unlink(missing_ok=True)
    return s


print("Usage before:")
for p in sorted(Path("/kaggle/working").iterdir()):
    print(f"  {gb(du(p)):>9}  {p}")
for p in sorted(OUT.iterdir()):
    print(f"      {gb(du(p)):>9}  {p.name}")

L = ledger_all(); freed = 0

# ---- 1. stale resume files: the best checkpoint or a ledger entry already exists ----
print("\n1) stale resume / temp files")
for f in list(CKPT.glob("*.last.pt")) + list(CKPT.glob("*.tmp")):
    rid = f.name.replace(".last.pt", "").replace(".tmp", "")
    if rid in L or (CKPT/f"{rid}.pt").exists():
        freed += rm(f, "run finished")

# ---- 2. caches no unfinished run still needs (rebuildable, never retraining) ----
print("\n2) caches not needed by any unfinished run")
need_tags, need_whole = {CFG.cache_tag()}, False          # base cache: analysis, eval-only, cross-corpus
for n, ss, ff in planned_jobs():
    if all(f"{n}__s{s}__f{k}" in L for s in ss for k in ff): continue
    sp = get_spec(n); cfg_ = replace(CFG, **sp["cfg_over"]) if sp["cfg_over"] else CFG
    if sp["kind"] == "whole": need_whole = True
    else: need_tags.add(cfg_.cache_tag())
for d in sorted(p for p in CACHE.iterdir() if p.is_dir()):
    if d.name.startswith("whole_r"):
        if not need_whole: freed += rm(d, "all whole_image runs done")
    elif d.name not in need_tags:
        freed += rm(d, "all runs using this tiling done")

# ---- 3. move the remaining caches out of the 20 GB quota ----
print("\n3) relocate remaining caches")
if MOVE_CACHE_TO_TMP and CACHE.resolve() != TMP_CACHE.resolve():
    for d in sorted(p for p in CACHE.iterdir() if p.is_dir()):
        s = du(d); print(f"  {'[dry] ' if DRY_RUN else ''}move   {gb(s):>9}  {d} -> {TMP_CACHE/d.name}")
        if not DRY_RUN:
            TMP_CACHE.mkdir(parents=True, exist_ok=True)
            shutil.move(str(d), str(TMP_CACHE/d.name)); freed += s
    if not DRY_RUN:
        CACHE = TMP_CACHE; CFG.cache_dir = str(TMP_CACHE)
        _ZCACHE.clear(); _WCACHE.clear()
        Z0 = get_cache(CFG)                                   # re-open from the new location
        print(f"  CACHE -> {CACHE}  (cache hit expected)")

# ---- 4. optional: checkpoints nothing downstream reloads ----
print("\n4) checkpoints of finished runs not reloaded downstream")
KEEP = set(CFG.exh_specs) | {"full", "doc_only"}             # exhaustive, eval-only, cross-corpus, figures
for f in sorted(CKPT.glob("*.pt")):
    if f.name.endswith(".last.pt"): continue
    rid = f.stem; name = rid.split("__")[0]
    if name in KEEP or rid not in L or not (PRED/f"{rid}.npz").exists(): continue
    if DROP_UNUSED_CKPTS: freed += rm(f, "predictions saved")
    else: print(f"  [kept] {gb(du(f)):>9}  {f.name}  (set DROP_UNUSED_CKPTS=True to remove)")

print(f"\n{'Would free' if DRY_RUN else 'Freed'}: {gb(freed)} | /kaggle/working now {gb(du('/kaggle/working'))}")
if MOVE_CACHE_TO_TMP and not DRY_RUN:
    print(f"NEXT SESSION: in the config cell set cache_dir='{TMP_CACHE}' in PRESETS['{CFG.dataset}'] "
          "(the cache is rebuilt there once, identically; no retraining).")